# DREAMwalk MSI: baseline and controlled experiments

Follow the investigation: **shared preparation → baseline → walk length → pp=1/0 → walk source → embedding trainer → semantic pp → teleport selector → NS pool → positive-output filter → MeSH2021 → combined variant → summary**. Each section displays its parameters, individual steps and results.

Select any local Python 3 kernel. Computation uses the separately installed DREAMwalk Python 3.8 environment. **Sections 7–8 use the MeSH2024 hierarchy with letter-category roots and new run names. To run those controls: Section 1 → Section 7 → Section 8. Section 8 requires Section 7's completed pp=0 evaluation.** Section 12 runs the combined MeSH2024 variant. Section 13 separates historical and revised-input results. Running every section is not required.

Computation cells reuse validated checkpoints; missing stages run only when explicitly started. To inspect existing results, run setup/parameter/display cells and skip computation cells. Save each new embedding before evaluation.

**Reproduction rule:** accept settings consistent with the paper and released code. Do not tune epochs, subsampling, windows or negative counts merely to match a reported score. Only the specified controls run.

Original upstream sources and historical experiment definitions remain available. Section 12 has separate inputs, binary, corpus and result paths. This code release excludes local working artifacts; see the repository README for required inputs and historical results.


## Repeated evaluation: 10 x 10 CV

Open [MSI_DREAMwalk_10x10_CV.ipynb](MSI_DREAMwalk_10x10_CV.ipynb) for ten seeds and ten folds per seed on the completed MeSH2024 Original/Modified embeddings. It reuses verified seed-0 results and starts new classifier fits only in Sections 7 and 8. The full 840-disease cohort and ten disjoint Link-level negative sets are used; embeddings stay fixed. Historical controls below are retained.

For category generalization, use [MSI_DREAMwalk_disease_area_split.ipynb](MSI_DREAMwalk_disease_area_split.ipynb). It retains both strict all-membership and fixed-primary-category split attempts, with their completed outputs.


## 1. Shared preparation

Configure paths, interpreter, inputs, logs and table display once. Use the reviewed molecular graph and saved seed-0 negatives. This notebook does not repeat raw preprocessing or mapping.


In [51]:
from pathlib import Path
from functools import partial
import sys

PROJECT_OVERRIDE = None
PYTHON_OVERRIDE = None

start = Path(PROJECT_OVERRIDE).expanduser().resolve() if PROJECT_OVERRIDE else Path.cwd().resolve()
ROOT = next((candidate
             for parent in [start, *start.parents]
             for candidate in [parent, parent/'00.DREAMwalk_reproduction',
                               parent/'00.Reproduction/00.DREAMwalk_reproduction']
             if (candidate/'src/dreamwalk_notebook_workflow.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('DREAMwalk not found; set PROJECT_OVERRIDE.')
PYTHON = Path(PYTHON_OVERRIDE).expanduser().resolve() if PYTHON_OVERRIDE else (
    ROOT/'environment/.conda-env/bin/python')
RESULTS = ROOT/'results'
sys.path.insert(0, str(ROOT/'src'))
from dreamwalk_notebook_workflow import (
    read_csv, read_json, summary_metrics, runner_constants, check_parameters,
    run_phase, table_complete, show_rows, show_csv,
)
# Share stage execution, logging and cancellation; keep experiment ordering visible below.
run = partial(run_phase, ROOT, PYTHON)

INPUTS = {
    'molecular_graph': ROOT/'data/processed/msi_graph.tsv',
    'node_types': ROOT/'data/processed/msi_nodetypes.tsv',
    'positives': ROOT/'data/processed/msi_positive_pairs.tsv',
    'negatives': RESULTS/'msi_repeat0_negative_pairs.csv',
}
for name, path in INPUTS.items():
    if not path.is_file():
        raise FileNotFoundError(f'Required input missing: {name}: {path}')
print('Project: ', ROOT)
print('Computation interpreter: ', PYTHON)
print('Shared inputs: ')
for name, path in INPUTS.items():
    print(' ', name, '→', path)


Project: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction
Computation interpreter: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/environment/.conda-env/bin/python
Shared inputs:
  molecular_graph → /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_graph.tsv
  node_types → /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_nodetypes.tsv
  positives → /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_positive_pairs.tsv
  negatives → /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_repeat0_negative_pairs.csv


### Shared experiment settings

| Setting | Value |
|---|---|
| Molecular graph | 29,959 nodes; 478,728 non-treatment input rows |
| Pairs / CV | 5,926 positives + 5,926 saved negatives; seed-0 ten-fold CV |
| Walks / node | 100; 2,995,900 total |
| p / q / EM iterations | 1 / 1 / at most 5 |
| Dimensions / window / workers | 128 / 4 / 4 |
| Pair features | Drug embedding minus disease embedding |
| Predictor | Released XGBoost settings; 500 trees, depth 6 |

Read actual constants from each runner and check the intended contrast. Parameters are defined by the implementation; notebook-only variables are not presented as working controls. New parameter experiments must update the corresponding section and runner and use a new run name.


In [52]:
# Check that controls retain these shared settings.
EXPECTED_SHARED_WALKS = {
    'seed': 0, 'num_walks': 100, 'p': 1, 'q': 1,
    'workers': 4, 'em_max_iter': 5,
}
show_rows([EXPECTED_SHARED_WALKS])


seed,num_walks,p,q,workers,em_max_iter
0,100,1,1,4,5


## 2. Baseline: read completed results

### 2.1 Semantic DREAMwalk

Local semantic seed 0: τ=0.3, length=10, pp=1, cutoff=0.4. Read the ten-fold evaluation of the same embedding/pairs, rather than the earlier five-fold evaluator.


In [3]:
semantic_config = read_json(RESULTS/'msi_repeat0_run_config.json')
semantic_settings = semantic_config['dreamwalk']
semantic_rows = read_csv(RESULTS/'diagnostic_repeat0_comparison.csv')
show_rows([{
    'tau': semantic_settings['tp_factor'],
    'walk_length': semantic_settings['walk_length'],
    'pp': 1,
    'cutoff': semantic_settings['cutoff'],
    'dimension': semantic_settings['dimension'],
    'window': semantic_settings['window_size'],
}])
show_rows([{
    'metric': row['metric'],
    'semantic_10fold': row['same_embedding_same_pairs_semantic_10fold'],
    'paper_semantic': row['paper_semantic'],
} for row in semantic_rows])


tau,walk_length,pp,cutoff,dimension,window
0.300000,10,1,0.400000,128,4


metric,semantic_10fold,paper_semantic
accuracy,0.803747,0.877098
auroc,0.884284,0.944060
aupr,0.893138,0.946767


### 2.2 No teleport: historical length=10

τ=0, length=10, pp=1. This historical baseline is the walk-length control. Its corpus differs from the later shared-corpus pp experiment; keep both results separate.


In [4]:
NO_TELEPORT_BASELINE = {'tp_factor': 0, 'walk_length': 10, 'pp': 1}
show_rows([NO_TELEPORT_BASELINE])
show_csv(RESULTS/'diagnostic_no_teleport_summary.csv')
historical_no_teleport = summary_metrics(RESULTS/'diagnostic_no_teleport_summary.csv')


diagnostic_no_teleport_summary.csv


tp_factor,walk_length,pp
0,10,1


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport,0,accuracy,0.786028,0.006942,0.772344,0.794266,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport,0,auroc,0.865565,0.008106,0.851170,0.879232,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport,0,aupr,0.875922,0.008023,0.860730,0.886676,10


## 3. Control 1: walk length

**Question: can the main text's length=4 explain the no-teleport gap?** Compare with historical length=10. Keep τ=0, pp=1, 100 walks/node, 128D, window 4 and the same ten-fold procedure.

Upstream passes the same `walk_length` to EM edge paths and final node walks. This control changes both lengths.


In [5]:
LENGTH4_RUN_NAME = 'msi_seed0_no_teleport_length4'
LENGTH4_RUNNER = 'run_no_teleport_length4.py'
LENGTH4_RESULTS = RESULTS/LENGTH4_RUN_NAME
LENGTH4_EMBEDDING = ROOT/'data/embeddings'/f'{LENGTH4_RUN_NAME}.pkl'

# Read runner constants; editing notebook descriptions does not change model settings.
length4_parameters = runner_constants(
    ROOT/'src'/LENGTH4_RUNNER, 'PARAMETERS')['PARAMETERS']
check_parameters(length4_parameters, {
    **EXPECTED_SHARED_WALKS, 'tp_factor': 0, 'walk_length': 4,
    'dimension': 128, 'window_size': 4,
})
show_rows([
    {'variant': 'Historical control', 'tau': 0, 'walk_length': 10, 'pp': 1},
    {'variant': 'Length-4 experiment',
     'tau': length4_parameters['tp_factor'],
     'walk_length': length4_parameters['walk_length'], 'pp': 1},
])
print('Embedding：', LENGTH4_EMBEDDING)
print('Results: ', LENGTH4_RESULTS)


Embedding： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_no_teleport_length4.pkl
Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_no_teleport_length4


variant,tau,walk_length,pp
Historical control,0,10,1
Length-4 experiment,0,4,1


### 3.1 Check environment and configuration

The runner checks Python/package pins and input/source fingerprints to prevent mixing checkpoints.


In [ ]:
run(LENGTH4_RUNNER, 'config', LENGTH4_RUN_NAME)

### 3.2 EM → walks → heterogeneous Skip-gram

Generate or reuse the length-4 embedding. Save it immediately before evaluation.


In [ ]:
run(LENGTH4_RUNNER, 'embedding', LENGTH4_RUN_NAME)
print('Embedding saved/reused: ', LENGTH4_EMBEDDING)


### 3.3 Ten-fold XGBoost


In [ ]:
run(LENGTH4_RUNNER, 'evaluate', LENGTH4_RUN_NAME)

### 3.4 Compare length=4 with length=10

Inspect `length4_minus_length10`. Keep the original historical control; the later shared-corpus pp=1 result is a separate execution.


In [6]:
show_csv(LENGTH4_RESULTS/'comparison_length4_vs_length10.csv')
show_csv(LENGTH4_RESULTS/'summary.csv')
length4_metrics = summary_metrics(LENGTH4_RESULTS/'summary.csv')
show_rows([{
    'comparison': 'length4 − historical length10',
    'AUROC_delta': length4_metrics['auroc'] - historical_no_teleport['auroc'],
    'AUPR_delta': length4_metrics['aupr'] - historical_no_teleport['aupr'],
}])


comparison_length4_vs_length10.csv
summary.csv


metric,no_teleport_length10,no_teleport_length4,length4_minus_length10,paper_no_teleport_approx,length4_minus_paper_approx
accuracy,0.786028,0.752196,-0.033832,—,—
auroc,0.865565,0.832682,-0.032883,0.905000,-0.072318
aupr,0.875922,0.843787,-0.032136,0.912000,-0.068213


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_length4,0,accuracy,0.752196,0.011345,0.729342,0.767932,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_length4,0,auroc,0.832682,0.012690,0.810112,0.849691,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_length4,0,aupr,0.843787,0.012107,0.828660,0.861442,10


comparison,AUROC_delta,AUPR_delta
length4 − historical length10,-0.032883,-0.032136


## 4. Control 2: Skip-gram negative sampling

**Question: what changes when the pp branch changes on fixed walks?** Generate one τ=0, length-10 corpus. Both pp branches use identical corpus bytes, token map, C++ binary and folds. Switching pp changes the whole NS branch, including rejection behavior. pp=0 is a diagnostic ablation; original DREAMwalk uses pp=1.


In [7]:
PP_RUN_NAME = 'msi_seed0_skipgram_pp_comparison'
PP_RUNNER = 'run_skipgram_pp_comparison.py'
PP_RESULTS = RESULTS/PP_RUN_NAME
PP_DATA = ROOT/'data/embeddings'/PP_RUN_NAME

pp_constants = runner_constants(ROOT/'src'/PP_RUNNER, 'WALKS', 'SG_ARGS')
pp_walk_parameters = pp_constants['WALKS']
check_parameters(pp_walk_parameters, {
    **EXPECTED_SHARED_WALKS, 'tp_factor': 0, 'walk_length': 10,
})
# Both C++ calls use this exact argument list.
pp_sg_args = dict(zip(pp_constants['SG_ARGS'][::2], pp_constants['SG_ARGS'][1::2]))
check_parameters(pp_sg_args, {
    '-size': '128', '-window': '4', '-iter': '1', '-threads': '4',
})
show_rows([
    {'variant': 'pp=1', **pp_walk_parameters, 'pp': 1},
    {'variant': 'pp=0', **pp_walk_parameters, 'pp': 0},
])
show_rows([pp_sg_args])
print('Shared corpus: ', PP_DATA/'walks.txt')
print('Results: ', PP_RESULTS)


Shared corpus: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/walks.txt
Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_skipgram_pp_comparison


variant,tp_factor,seed,num_walks,walk_length,p,q,workers,em_max_iter,pp
pp=1,0,0,100,10,1,1,4,5,1
pp=0,0,0,100,10,1,1,4,5,0


-min-count,-size,-iter,-window,-threads
0,128,1,4,4


### 4.1 Check configuration → compile one C++ binary → shared walks

Prepare one common input. Validate and reuse completed binary/corpus checkpoints.


In [ ]:
run(PP_RUNNER, 'config', PP_RUN_NAME)

In [ ]:
run(PP_RUNNER, 'compile', PP_RUN_NAME)

In [ ]:
run(PP_RUNNER, 'walks', PP_RUN_NAME)

### 4.2 pp=1: train, save immediately, evaluate ten folds


In [ ]:
run(PP_RUNNER, 'pp1', PP_RUN_NAME)
print('pp=1 embedding saved/reused: ', PP_DATA/'pp1.pkl')


In [ ]:
run(PP_RUNNER, 'evaluate-pp1', PP_RUN_NAME)

In [8]:
show_csv(PP_RESULTS/'pp1_summary.csv')

pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,accuracy,0.788137,0.011062,0.771308,0.808439,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,auroc,0.866120,0.007938,0.855043,0.880914,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,aupr,0.874306,0.010921,0.860393,0.892279,10


### 4.3 pp=0: train and evaluate on the same corpus


In [ ]:
run(PP_RUNNER, 'pp0', PP_RUN_NAME)
print('pp=0 embedding saved/reused: ', PP_DATA/'pp0.pkl')


In [ ]:
run(PP_RUNNER, 'evaluate-pp0', PP_RUN_NAME)

In [9]:
show_csv(PP_RESULTS/'pp0_summary.csv')

pp0_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp0,0,accuracy,0.803241,0.011556,0.782462,0.814503,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp0,0,auroc,0.881752,0.012309,0.857955,0.897281,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp0,0,aupr,0.889735,0.012474,0.871633,0.905549,10


### 4.4 Compare pp=0 and pp=1

Read an existing hash-validated comparison. Otherwise the runner checks shared corpus, binary, map and folds before generating it.


In [10]:
if not table_complete(
    PP_RESULTS/'pp_comparison.csv',
    PP_RESULTS/'comparison_complete.json',
    PP_RESULTS/'run_config.json',
):
    run(PP_RUNNER, 'compare', PP_RUN_NAME)
show_csv(PP_RESULTS/'pp_comparison.csv')
show_csv(PP_RESULTS/'paired_fold_deltas.csv')


pp_comparison.csv
paired_fold_deltas.csv


metric,same_walks_pp1,same_walks_pp0,pp0_minus_pp1,historical_length10_pp1,paper_no_teleport_approx
accuracy,0.788137,0.803241,0.015103,0.786028,—
auroc,0.866120,0.881752,0.015632,0.865565,0.905000
aupr,0.874306,0.889735,0.015430,0.875922,0.912000


fold,accuracy_pp0_minus_pp1,auroc_pp0_minus_pp1,aupr_pp0_minus_pp1
1,-0.000843,-0.007064,-0.003309
2,0.026138,0.022667,0.017311
3,0.037975,0.027360,0.027293
4,0.015190,0.013630,0.007965
5,0.010970,0.012619,0.009879
6,0.018565,0.016027,0.022569
7,0.005063,0.016368,0.013270
8,0.007595,0.016177,0.017032
9,0.013502,0.020156,0.024537
10,0.016878,0.018376,0.017752


## 5. Control 3: edge2vec walks with DREAMwalk training/evaluation

Run Section 1, then this section. Completed earlier experiments need not be repeated.

**Question: what changes when saved edge2vec walks replace no-teleport DREAMwalk walks?** Reuse **2,995,900 length-10 edge2vec walks**, without new EM, sampling or semantic mapping. Decode numeric IDs to real nodes, then encode them with the same DREAMwalk token map. Train the same C++ binary with **pp=1**.

The reference is **DREAMwalk shared walks → pp=1 → XGBoost** (AUROC 0.866120), rather than standalone edge2vec's Gensim/LR score or pp=0. Keep Skip-gram settings, type prefixes, XGBoost, negatives and folds fixed. This compares two saved corpora; it cannot isolate individual EM/sampling steps.


In [ ]:
EDGE_WALKS_RUN_NAME = 'msi_seed0_edge2vec_walks_pp1'
EDGE_WALKS_RUNNER = 'run_edge2vec_walks_control.py'
EDGE_WALKS_RESULTS = RESULTS/EDGE_WALKS_RUN_NAME
EDGE_WALKS_DATA = ROOT/'data/embeddings'/EDGE_WALKS_RUN_NAME
EDGE_SOURCE_ROOT = ROOT.parent/'03.edge2vec_reproduction'
EDGE_SOURCE_CORPUS = EDGE_SOURCE_ROOT/'data/msi_seed0/walks.txt'
EDGE_SOURCE_NODES = EDGE_SOURCE_ROOT/'data/msi_seed0/nodes.json'
# Match the runner to its completed historical pp=1 reference.
EDGE_WALKS_REFERENCE_DATA = ROOT/'data/embeddings/msi_seed0_skipgram_pp_comparison'

# Read actual runner parameters instead of displaying unused controls.
edge_walk_settings = runner_constants(
    ROOT/'src'/EDGE_WALKS_RUNNER, 'RUN_SETTINGS')['RUN_SETTINGS']
check_parameters(edge_walk_settings, {
    'pp': 1, 'seed': 0, 'num_walks': 100, 'walk_length': 10,
    'dimension': 128, 'window_size': 4, 'workers': 4, 'epochs': 1,
    'folds': 10, 'cv_seed': 0, 'source_walks': 2995900, 'source_nodes': 29959,
})
# Reuse the original pp runner SG_ARGS and run pp=1 only.
edge_sg_constants = runner_constants(ROOT/'src/run_skipgram_pp_comparison.py', 'SG_ARGS')['SG_ARGS']
edge_sg_args = dict(zip(edge_sg_constants[::2], edge_sg_constants[1::2]))
check_parameters(edge_sg_args, {
    '-size': '128', '-window': '4', '-iter': '1', '-threads': '4',
})
show_rows([edge_walk_settings])
show_rows([edge_sg_args])
print('Completed edge2vec corpus: ', EDGE_SOURCE_CORPUS)
print('Numeric ID → real node：', EDGE_SOURCE_NODES)
print('Reused token map: ', EDGE_WALKS_REFERENCE_DATA/'token_to_node.json')
print('Reused C++ binary: ', EDGE_WALKS_REFERENCE_DATA/'HetSG')
print('Embedding for this section: ', EDGE_WALKS_DATA/'pp1.pkl')
print('Results for this section: ', EDGE_WALKS_RESULTS)

Completed edge2vec corpus: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/03.edge2vec_reproduction/data/msi_seed0/walks.txt
Numeric ID → real node： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/03.edge2vec_reproduction/data/msi_seed0/nodes.json
Reused token map: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/token_to_node.json
Reused C++ binary: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/HetSG
Section embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_edge2vec_walks_pp1/pp1.pkl
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_edge2vec_walks_pp1


pp,seed,num_walks,walk_length,dimension,window_size,workers,epochs,folds,cv_seed,source_walks,source_nodes
1,0,100,10,128,4,4,1,10,0,2995900,29959


-min-count,-size,-iter,-window,-threads
0,128,1,4,4


### 5.1 Check environment, shared inputs and provenance

Verify completed edge2vec corpus/node map, DREAMwalk pp=1 reference, graph and pairs. Save a separate configuration fingerprint.


In [ ]:
run(EDGE_WALKS_RUNNER, 'config', EDGE_WALKS_RUN_NAME)

### 5.2 Re-encode saved walks; reuse map and binary

Check source completion marker and SHA-256. Convert **edge2vec numeric ID → real node → DREAMwalk type-prefixed token** line by line. Preserve node order, walk order and paths. Copy the existing map/binary without recompiling or sampling; record both corpus hashes.


In [ ]:
run(EDGE_WALKS_RUNNER, 'prepare', EDGE_WALKS_RUN_NAME)

### 5.3 Original heterogeneous Skip-gram: pp=1

Use 128D, window 4, one epoch and four workers, matching the shared-walks pp=1 control. Train one embedding and save it before evaluation.


In [ ]:
run(EDGE_WALKS_RUNNER, 'embedding', EDGE_WALKS_RUN_NAME)
print('Embedding saved/reused: ', EDGE_WALKS_DATA/'pp1.pkl')

### 5.4 Same pairs and folds: ten-fold XGBoost

Reuse the DREAMwalk evaluator, saved seed-0 negatives and folds, with `drug_embedding - disease_embedding` features. Do not use standalone edge2vec's LR evaluator.


In [ ]:
run(EDGE_WALKS_RUNNER, 'evaluate', EDGE_WALKS_RUN_NAME)

In [ ]:
show_csv(EDGE_WALKS_RESULTS/'pp1_summary.csv')

pp1_summary.csv
No saved results yet.


### 5.5 Compare walk sources

Inspect `edge2vec_minus_dreamwalk` and paired fold deltas: **same DREAMwalk Skip-gram and XGBoost, different saved walk source**. The paper's no-teleport result is an external reference; standalone edge2vec is not the baseline here.


In [ ]:
if not table_complete(
    EDGE_WALKS_RESULTS/'walk_source_comparison.csv',
    EDGE_WALKS_RESULTS/'comparison_complete.json',
    EDGE_WALKS_RESULTS/'run_config.json',
):
    run(EDGE_WALKS_RUNNER, 'compare', EDGE_WALKS_RUN_NAME)
show_csv(EDGE_WALKS_RESULTS/'walk_source_comparison.csv')
show_csv(EDGE_WALKS_RESULTS/'paired_fold_deltas.csv')

## 6. Control 4: compare embedding trainers on the same edge2vec walks

Run Section 1, then this section; Sections 2–5 need not be repeated.

**Question: what changes when the saved Gensim embedding replaces the DREAMwalk C++ embedding under the same XGBoost evaluation?**

| Branch | Walk corpus | Embeddings | Evaluation |
|---|---|---|---|
| Completed reference | Same edge2vec corpus | DREAMwalk C++, pp=1 | DREAMwalk XGBoost |
| This section | Same edge2vec corpus | Saved Gensim Skip-gram | Same XGBoost |

Read standalone edge2vec's saved `msi_edge2vec_seed0.pkl`. Generate no new walks or embeddings. Keep labeled pairs, subtraction features, XGBoost and fold IDs fixed.

This replaces the **whole embedding stage**: NS distribution, implementation and initialization can all differ. Gensim has no corresponding pp switch and is not labeled pp=0. Original DREAMwalk retains pp=1.


In [3]:
GENSIM_RUN_NAME = 'msi_seed0_edge2vec_gensim_xgb'
GENSIM_RUNNER = 'run_gensim_embedding_control.py'
GENSIM_RESULTS = RESULTS/GENSIM_RUN_NAME
GENSIM_SOURCE_ROOT = ROOT.parent/'03.edge2vec_reproduction'
GENSIM_SOURCE_EMBEDDING = GENSIM_SOURCE_ROOT/'data/msi_seed0/msi_edge2vec_seed0.pkl'
GENSIM_REFERENCE_RESULTS = RESULTS/'msi_seed0_edge2vec_walks_pp1'

# Read runner settings; this section does not generate walks or train Skip-gram.
gensim_control_settings = runner_constants(
    ROOT/'src'/GENSIM_RUNNER, 'RUN_SETTINGS')['RUN_SETTINGS']
check_parameters(gensim_control_settings, {
    'seed': 0, 'dimension': 128, 'folds': 10, 'cv_seed': 0,
    'source_walks': 2995900, 'source_nodes': 29959,
})
show_rows([gensim_control_settings])

# Display the completed embedding settings; these variables do not launch new training.
gensim_source_config = read_json(GENSIM_SOURCE_ROOT/'results/msi_seed0/run_config.json')
gensim_training = gensim_source_config['config']
check_parameters(gensim_training, {
    'sg': 1, 'hs': 0, 'vector_size': 128, 'window': 4,
    'workers': 4, 'epochs': 1, 'negative': 5,
    'num_walks': 100, 'walk_length': 10,
})
gensim_setting_names = (
    'sg', 'hs', 'vector_size', 'window', 'workers', 'epochs', 'negative',
    'sample', 'alpha', 'min_alpha', 'ns_exponent', 'min_count',
    'sorted_vocab', 'batch_words', 'shrink_windows',
)
show_rows([{name: gensim_training[name] for name in gensim_setting_names}])
reference_xgb = read_json(GENSIM_REFERENCE_RESULTS/'run_config.json')['xgboost']
show_rows([reference_xgb])
show_rows([
    {'Embedding trainer': 'C++ heterogeneous Skip-gram', 'pp': 1,
     'Negative sampling': 'node-type constrained; released pp=1 branch'},
    {'Embedding trainer': 'Gensim Skip-gram', 'pp': None,
     'Negative sampling': 'ordinary vocabulary NS; negative=5, exponent=0.75'},
])
print('Saved Gensim embedding: ', GENSIM_SOURCE_EMBEDDING)
print('Completed C++ pp=1 + XGBoost reference: ', GENSIM_REFERENCE_RESULTS)
print('Results for this section: ', GENSIM_RESULTS)


seed,dimension,folds,cv_seed,source_walks,source_nodes
0,128,10,0,2995900,29959


sg,hs,vector_size,window,workers,epochs,negative,sample,alpha,min_alpha,ns_exponent,min_count,sorted_vocab,batch_words,shrink_windows
1,0,128,4,4,1,5,0.001000,0.025000,0.000100,0.750000,0,1,10000,True


base_score,booster,eval_metric,objective,gamma,learning_rate,max_depth,n_estimators,tree_method,min_child_weight,subsample,colsample_bytree,scale_pos_weight,max_delta_step,seed,n_jobs
0.500000,gbtree,error,binary:logistic,0,0.100000,6,500,auto,4,0.800000,0.900000,1,1,0,4


Embedding trainer,pp,Negative sampling
C++ heterogeneous Skip-gram,1,node-type constrained; released pp=1 branch
Gensim Skip-gram,—,"ordinary vocabulary NS; negative=5, exponent=0.75"


Saved Gensim embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/03.edge2vec_reproduction/data/msi_seed0/msi_edge2vec_seed0.pkl
Completed C++ pp=1 + XGBoost reference: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_edge2vec_walks_pp1
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_edge2vec_gensim_xgb


### 6.1 Check saved embedding, corpus and reference

Record hashes/provenance of the Gensim embedding, original walks, shared graph/pairs and completed C++ reference. Check the shared corpus/downstream settings without training.


In [4]:
run(GENSIM_RUNNER, 'config', GENSIM_RUN_NAME)

Running: run_gensim_embedding_control.py config | msi_seed0_edge2vec_gensim_xgb
{
  "settings": {
    "seed": 0,
    "dimension": 128,
    "folds": 10,
    "cv_seed": 0,
    "source_walks": 2995900,
    "source_nodes": 29959
  },
  "xgboost": {
    "base_score": 0.5,
    "booster": "gbtree",
    "eval_metric": "error",
    "objective": "binary:logistic",
    "gamma": 0,
    "learning_rate": 0.1,
    "max_depth": 6,
    "n_estimators": 500,
    "tree_method": "auto",
    "min_child_weight": 4,
    "subsample": 0.8,
    "colsample_bytree": 0.9,
    "scale_pos_weight": 1,
    "max_delta_step": 1,
    "seed": 0,
    "n_jobs": 4
  },
  "feature": "drug_embedding - disease_embedding",
  "sources": {
    "inputs": {
      "graph": "8285845165826358275bc3dc379608a4e2de656b5398b5701123810eb51c61da",
      "nodetypes": "4e6898e31e3a06eb4d374b817c2a39cb42250534b5e6d46c0290fd0d1064e072",
      "positives": "5e8a402186092d94a5c576b754b12bb93ad90e4e51f851c20bba6e928bb7a5a4",
      "negatives": "f17b

### 6.2 Read Gensim embeddings; run the same ten-fold XGBoost

Read real-node vectors and construct subtraction features. Reuse the evaluator and negatives; check generated fold IDs against the reference before saving summaries and fold results. Only downstream XGBoost is fitted.


In [5]:
run(GENSIM_RUNNER, 'evaluate', GENSIM_RUN_NAME)

Running: run_gensim_embedding_control.py evaluate | msi_seed0_edge2vec_gensim_xgb
Evaluating saved Gensim vectors with the unchanged DREAMwalk XGBoost evaluator
no_teleport negative seed 0 fold 1 {'accuracy': 0.779089, 'auroc': 0.86242, 'aupr': 0.877406}
no_teleport negative seed 0 fold 2 {'accuracy': 0.814503, 'auroc': 0.889697, 'aupr': 0.895603}
no_teleport negative seed 0 fold 3 {'accuracy': 0.810127, 'auroc': 0.890989, 'aupr': 0.898679}
no_teleport negative seed 0 fold 4 {'accuracy': 0.802532, 'auroc': 0.879059, 'aupr': 0.890946}
no_teleport negative seed 0 fold 5 {'accuracy': 0.803376, 'auroc': 0.878566, 'aupr': 0.895966}
no_teleport negative seed 0 fold 6 {'accuracy': 0.796624, 'auroc': 0.871622, 'aupr': 0.886214}
no_teleport negative seed 0 fold 7 {'accuracy': 0.783966, 'auroc': 0.876897, 'aupr': 0.891685}
no_teleport negative seed 0 fold 8 {'accuracy': 0.805063, 'auroc': 0.887075, 'aupr': 0.891734}
no_teleport negative seed 0 fold 9 {'accuracy': 0.806751, 'auroc': 0.872923, 'au

In [6]:
show_csv(GENSIM_RESULTS/'summary.csv')

summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,edge2vec_gensim_xgb,0,accuracy,0.800372,0.011101,0.779089,0.814503,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,edge2vec_gensim_xgb,0,auroc,0.878697,0.008787,0.862420,0.890989,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,edge2vec_gensim_xgb,0,aupr,0.889029,0.007299,0.877406,0.898679,10


### 6.3 Compare Gensim with C++ pp=1

Inspect `gensim_minus_cpp_pp1` and paired fold deltas. Both rows use DREAMwalk XGBoost. The contrast measures the complete embedding-stage substitution, rather than NS or initialization alone.


In [7]:
if not table_complete(
    GENSIM_RESULTS/'embedding_trainer_comparison.csv',
    GENSIM_RESULTS/'comparison_complete.json',
    GENSIM_RESULTS/'run_config.json',
):
    run(GENSIM_RUNNER, 'compare', GENSIM_RUN_NAME)
show_csv(GENSIM_RESULTS/'embedding_trainer_comparison.csv')
show_csv(GENSIM_RESULTS/'paired_fold_deltas.csv')


Running: run_gensim_embedding_control.py compare | msi_seed0_edge2vec_gensim_xgb
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_edge2vec_gensim_xgb/summary.csv
[
  {
    "metric": "accuracy",
    "cpp_pp1": 0.788728484926107,
    "gensim_sg": 0.8003718487843405,
    "gensim_minus_cpp_pp1": 0.011643363858233502
  },
  {
    "metric": "auroc",
    "cpp_pp1": 0.8673203846618246,
    "gensim_sg": 0.8786968446461471,
    "gensim_minus_cpp_pp1": 0.011376459984322462
  },
  {
    "metric": "aupr",
    "cpp_pp1": 0.8773890890922059,
    "gensim_sg": 0.8890288569286883,
    "gensim_minus_cpp_pp1": 0.011639767836482351
  }
]
embedding_trainer_comparison.csv


metric,cpp_pp1,gensim_sg,gensim_minus_cpp_pp1
accuracy,0.788728,0.800372,0.011643
auroc,0.867320,0.878697,0.011376
aupr,0.877389,0.889029,0.011640


paired_fold_deltas.csv


fold,metric,cpp_pp1,gensim_sg,gensim_minus_cpp_pp1
1,accuracy,0.772344,0.779089,0.006745
1,auroc,0.857082,0.862420,0.005338
1,aupr,0.872127,0.877406,0.005278
2,accuracy,0.795110,0.814503,0.019393
2,auroc,0.872683,0.889697,0.017014
2,aupr,0.879586,0.895603,0.016017
3,accuracy,0.799156,0.810127,0.010970
3,auroc,0.880062,0.890989,0.010927
3,aupr,0.888089,0.898679,0.010590
4,accuracy,0.777215,0.802532,0.025316


## 7. Control 5: pp=1 versus pp=0 on revised MeSH2024 semantic walks

**Run Section 1, then this section. Sections 2–6 need not be repeated.**

**Question: with semantic teleportation (τ=0.3), does pp=1 still perform below pp=0 on the same corpus?** Section 4 disables semantic transitions with τ=0. This section uses Section 12's **MeSH2024 hierarchy with letter-category roots**, cutoff 0.4, a separate run name and a new corpus. Run original EM and save **2,995,900 length-10 walks** once for both branches. Historical-input results remain separate in Section 13.

| Branch | Corpus | Skip-gram | Downstream |
|---|---|---|---|
| pp=1 | Same τ=0.3 semantic walks | Original C++ heterogeneous NS | Same XGBoost, negatives and folds |
| pp=0 | Same corpus | Original C++ ordinary NS branch | Same evaluation |

Reuse Section 4's token map and C++ binary without recompiling. Do not change teleport selection, NS pools or RNG startup. pp=0 is a diagnostic ablation. Save each embedding before evaluation.

Only the semantic input changes relative to the historical version of this section. Do not combine the diagnostic changes from Sections 8–10 here.


In [53]:
SEMANTIC_PP_RUN_NAME = 'msi_seed0_semantic_pp_mesh2024_roots'
SEMANTIC_PP_RUNNER = 'run_semantic_mesh2024_controls.py'
SEMANTIC_PP_RESULTS = RESULTS/SEMANTIC_PP_RUN_NAME
SEMANTIC_PP_DATA = ROOT/'data/embeddings'/SEMANTIC_PP_RUN_NAME
SEMANTIC_PP_REFERENCE_DATA = ROOT/'data/embeddings/msi_seed0_skipgram_pp_comparison'

# Read runner constants without depending on variables from Sections 2–6.
semantic_pp_constants = runner_constants(
    ROOT/'src'/SEMANTIC_PP_RUNNER, 'WALK_SETTINGS', 'SEMANTIC_SETTINGS')
semantic_pp_walk_settings = semantic_pp_constants['WALK_SETTINGS']
semantic_pp_semantic_settings = semantic_pp_constants['SEMANTIC_SETTINGS']
check_parameters(semantic_pp_walk_settings, {
    'tp_factor': 0.3, 'seed': 0, 'num_walks': 100, 'walk_length': 10,
    'p': 1, 'q': 1, 'workers': 4, 'em_max_iter': 5,
})
check_parameters(semantic_pp_semantic_settings, {'mesh_year': 2024, 'cutoff': 0.4, 'letter_roots': True})

# Both branches use the same SG_ARGS; only -pp differs.
semantic_pp_sg_constants = runner_constants(
    ROOT/'src/run_skipgram_pp_comparison.py', 'SG_ARGS')['SG_ARGS']
semantic_pp_sg_args = dict(zip(semantic_pp_sg_constants[::2], semantic_pp_sg_constants[1::2]))
check_parameters(semantic_pp_sg_args, {
    '-size': '128', '-window': '4', '-iter': '1', '-threads': '4',
})
show_rows([
    {'variant': 'Semantic pp=1', **semantic_pp_walk_settings,
     **semantic_pp_semantic_settings, 'pp': 1},
    {'variant': 'Semantic pp=0', **semantic_pp_walk_settings,
     **semantic_pp_semantic_settings, 'pp': 0},
])
show_rows([semantic_pp_sg_args])
print('Shared semantic corpus: ', SEMANTIC_PP_DATA/'walks.txt')
print('Reused token map: ', SEMANTIC_PP_REFERENCE_DATA/'token_to_node.json')
print('Reused C++ binary: ', SEMANTIC_PP_REFERENCE_DATA/'HetSG')
print('Results for this section: ', SEMANTIC_PP_RESULTS)

SEMANTIC_INPUT_AUDITS = RESULTS/'msi_mesh2024_demo_roots_inputs'
print('Hierarchy for this section: ', ROOT/'data/processed/msi_mesh2024_demo_roots_hierarchy.csv')
print('Semantic graph for this section: ', ROOT/'data/processed/msi_mesh2024_demo_roots_similarity.tsv')


variant,tp_factor,seed,num_walks,walk_length,p,q,workers,em_max_iter,mesh_year,cutoff,letter_roots,pp
Semantic pp=1,0.300000,0,100,10,1,1,4,5,2024,0.400000,True,1
Semantic pp=0,0.300000,0,100,10,1,1,4,5,2024,0.400000,True,0


-min-count,-size,-iter,-window,-threads
0,128,1,4,4


Shared semantic corpus: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reused token map: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/token_to_node.json
Reused C++ binary: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/HetSG
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_semantic_pp_mesh2024_roots
Section hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_mesh2024_demo_roots_hierarchy.csv
Section semantic graph: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_mesh20

### 7.1 Check inputs, environment and configuration

Validate/reuse Section 12's MeSH2024 hierarchy and similarity, building only missing outputs. Record molecular inputs, seed-0 pairs, source and original binary identities.


In [54]:
run(SEMANTIC_PP_RUNNER, 'hierarchy', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py hierarchy | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
{
  "schema": 1,
  "stage": "hierarchy",
  "created_utc": "2026-10-18T12:01:20+00:00",
  "input_identity": {
    "schema": 1,
    "mesh_year": 2024,
    "cutoff": 0.4,
    "directed": false,
    "weighted": true,
    "upstream_commit": "2f93723c3fbfa899d0f23c02e3459de8bfac0581",
    "hierarchy_strategy": "descriptor_UI_plus_MeSH_tree_position_and_demo_letter_categories",
    "category_policy": "every_used_first_letter_plus_dis_suffix_before_disease_root",
    "mapping_policy": "same_direct_unique_descriptor_policy_revalidated_against_2024",
    "sources": {
      "mesh": {
        "path": "data/mesh/2024/d2024.bin",
        "sha256": "c96077dcfda566b010af0444f9f928f85e09c928fa88f17bcb7c0a

In [55]:
run(SEMANTIC_PP_RUNNER, 'similarity', SEMANTIC_PP_RUN_NAME)
show_csv(SEMANTIC_INPUT_AUDITS/'similarity_summary.csv')


Running: run_semantic_mesh2024_controls.py similarity | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
{
  "schema": 1,
  "stage": "similarity",
  "created_utc": "2026-10-18T12:01:36+00:00",
  "input_identity": {
    "schema": 1,
    "mesh_year": 2024,
    "cutoff": 0.4,
    "directed": false,
    "weighted": true,
    "upstream_commit": "2f93723c3fbfa899d0f23c02e3459de8bfac0581",
    "hierarchy_strategy": "descriptor_UI_plus_MeSH_tree_position_and_demo_letter_categories",
    "category_policy": "every_used_first_letter_plus_dis_suffix_before_disease_root",
    "mapping_policy": 

semantic_type,retained_undirected_pairs,nodes_with_semantic_edges,cutoff,saved_edge_directionality
drug,27530,1326,0.400000,one_row_per_undirected_pair
disease,25765,746,0.400000,one_row_per_undirected_pair


In [56]:
run(SEMANTIC_PP_RUNNER, 'config', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py config | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
{
  "walks": {
    "tp_factor": 0.3,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "p": 1,
    "q": 1,
    "workers": 4,
    "em_max_iter": 5
  },
  "semantic": {
    "mesh_year": 2024,
    "cutoff": 0.4,
    "letter_roots": true
  },
  "selector": "upstream",
  "control": "semantic",
  "skipgram_args": [
    "-min-count",
    "0",
    "-size",
    "128",
    "-iter",
    "1",
    "-window",
    "4",
    "-threads",
    "4"
  ],
  "xgboost": {
    "base_score": 0.5

### 7.2 Reuse the original token map and C++ binary

Copy exact files from the completed no-teleport pp control and record hashes. Retain identical type-prefixed tokens without recompilation.


In [57]:
run(SEMANTIC_PP_RUNNER, 'prepare', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py prepare | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Original binary, token map and fixed folds ready; control: semantic


### 7.3 One EM run → one semantic walk corpus

Use original EM/walk generation: τ=0.3, cutoff=0.4, 100 walks/node, length=10. Save the corpus and completion marker once. Both pp branches reuse that file without resampling.


In [58]:
run(SEMANTIC_PP_RUNNER, 'walks', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py walks | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
random seed with 0
CORRECTED MESH2024 CORPUS SAVED: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt


### 7.4 pp=1: original training, immediate saving, ten-fold evaluation

Use original DREAMwalk NS. Save embeddings before starting XGBoost.


In [59]:
run(SEMANTIC_PP_RUNNER, 'pp1', SEMANTIC_PP_RUN_NAME)
print('pp=1 embedding saved/reused: ', SEMANTIC_PP_DATA/'pp1.pkl')


Running: run_semantic_mesh2024_controls.py pp1 | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/

In [60]:
run(SEMANTIC_PP_RUNNER, 'evaluate-pp1', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py evaluate-pp1 | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduct

In [61]:
show_csv(SEMANTIC_PP_RESULTS/'pp1_summary.csv')


pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp1,0,accuracy,0.802736,0.019047,0.763912,0.822785,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp1,0,auroc,0.882394,0.013162,0.852555,0.893675,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp1,0,aupr,0.889991,0.010841,0.867042,0.904367,10


### 7.5 pp=0: train and evaluate on the same corpus

Use identical corpus bytes, map, binary, negatives and folds. Train each branch once without additional parameter combinations.


In [62]:
run(SEMANTIC_PP_RUNNER, 'pp0', SEMANTIC_PP_RUN_NAME)
print('pp=0 embedding saved/reused: ', SEMANTIC_PP_DATA/'pp0.pkl')


Running: run_semantic_mesh2024_controls.py pp0 | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/

In [63]:
run(SEMANTIC_PP_RUNNER, 'evaluate-pp0', SEMANTIC_PP_RUN_NAME)


Running: run_semantic_mesh2024_controls.py evaluate-pp0 | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduct

In [64]:
show_csv(SEMANTIC_PP_RESULTS/'pp0_summary.csv')


pp0_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,accuracy,0.809316,0.008050,0.792580,0.819409,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,auroc,0.892104,0.010099,0.871847,0.903323,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,aupr,0.898156,0.009165,0.879190,0.911426,10


### 7.6 Main contrast: semantic pp=1 − pp=0

Inspect `pp1_minus_pp0` and fold directions. Positive values favor pp=1; negative values favor pp=0.

`pp_by_teleport.csv` also compares the τ=0 and τ=0.3 pp effects; `pp_effect_difference` subtracts the former from the latter. **Branches share a corpus within each τ; different τ settings have separately sampled corpora.** This describes local effects without establishing a general interaction mechanism.


In [65]:
if not table_complete(
    SEMANTIC_PP_RESULTS/'pp_comparison.csv',
    SEMANTIC_PP_RESULTS/'comparison_complete.json',
    SEMANTIC_PP_RESULTS/'run_config.json',
):
    run(SEMANTIC_PP_RUNNER, 'compare', SEMANTIC_PP_RUN_NAME)
show_csv(SEMANTIC_PP_RESULTS/'pp_comparison.csv')
show_csv(SEMANTIC_PP_RESULTS/'paired_fold_deltas.csv')
show_csv(SEMANTIC_PP_RESULTS/'pp_by_teleport.csv')


Running: run_semantic_mesh2024_controls.py compare | msi_seed0_semantic_pp_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/r

metric,semantic_pp1,semantic_pp0,pp1_minus_pp0
accuracy,0.802736,0.809316,-0.006579
auroc,0.882394,0.892104,-0.009710
aupr,0.889991,0.898156,-0.008165


paired_fold_deltas.csv


fold,accuracy_pp1_minus_pp0,auroc_pp1_minus_pp0,aupr_pp1_minus_pp0
1,-0.028668,-0.019292,-0.012147
2,-0.007589,-0.008952,-0.004867
3,0.000000,-0.013758,-0.014259
4,0.011814,-0.010787,-0.010244
5,0.001688,0.011135,0.010905
6,-0.031224,-0.026181,-0.022566
7,-0.016878,-0.011935,-0.007376
8,-0.018565,-0.017237,-0.015083
9,0.020253,0.009961,-0.003818
10,0.003376,-0.010050,-0.002190


pp_by_teleport.csv


metric,no_teleport_pp1,no_teleport_pp0,no_teleport_pp1_minus_pp0,semantic_pp1,semantic_pp0,semantic_pp1_minus_pp0,pp_effect_difference
accuracy,0.788137,0.803241,-0.015103,0.802736,0.809316,-0.006579,0.008524
auroc,0.866120,0.881752,-0.015632,0.882394,0.892104,-0.009710,0.005922
aupr,0.874306,0.889735,-0.015430,0.889991,0.898156,-0.008165,0.007265


## 8. Control 6: teleport selector on revised MeSH2024 inputs, pp=0

**Run Section 1 and finish revised Section 7's walks and pp=0 evaluation before this section. Other sections need not be run.**

**Question: does returning at the first cumulative-weight hit change performance?**

| Branch | Selector | Skip-gram | Walks/evaluation |
|---|---|---|---|
| Completed baseline | Later neighbors can overwrite a selected target | Original C++, pp=0 | Revised Section 7 baseline |
| Diagnostic | Return at the first hit | Same binary, pp=0 | New walks; same XGBoost, pairs and folds |

Keep τ=0.3, cutoff=0.4, 100 walks/node, length 10, 128D, window 4 and four workers. Reuse revised Section 7's **EM matrix, token map, binary and original-selector baseline**. Generate walks only for the diagnostic selector; do not rerun EM or baseline.

Prevent target overwriting after a hit. Retain candidate order, shuffle and random-draw behavior. Install/audit the selector inside each worker. Corpora differ; this is not a replay with identical random draws.

This is a **DIAGNOSTIC IMPLEMENTATION CHANGE**, without a claim about which implementation generated the paper results. Keep historical and revised-input controls separate.


In [66]:
TELEPORT_RUN_NAME = 'msi_seed0_teleport_mesh2024_roots'
TELEPORT_RUNNER = 'run_teleport_mesh2024_control.py'
TELEPORT_RESULTS = RESULTS/TELEPORT_RUN_NAME
TELEPORT_DATA = ROOT/'data/embeddings'/TELEPORT_RUN_NAME
TELEPORT_REFERENCE_RESULTS = RESULTS/'msi_seed0_semantic_pp_mesh2024_roots'
TELEPORT_REFERENCE_DATA = ROOT/'data/embeddings/msi_seed0_semantic_pp_mesh2024_roots'

teleport_settings = runner_constants(ROOT/'src'/TELEPORT_RUNNER, 'RUN_SETTINGS')['RUN_SETTINGS']
check_parameters(teleport_settings, {
    'pp': 0, 'tp_factor': 0.3, 'seed': 0, 'num_walks': 100,
    'walk_length': 10, 'dimension': 128, 'window_size': 4,
    'epochs': 1, 'workers': 4, 'folds': 10, 'cv_seed': 0,
})
show_rows([teleport_settings])
print('Completed original-selector pp=0 baseline: ')
show_csv(TELEPORT_REFERENCE_RESULTS/'pp0_summary.csv')
print('Reused EM matrix: ', TELEPORT_REFERENCE_RESULTS/'edge_transition_matrix.json')
print('New walks: ', TELEPORT_DATA/'walks.txt')
print('New embedding: ', TELEPORT_DATA/'pp0.pkl')
print('Results for this section: ', TELEPORT_RESULTS)

pp,tp_factor,seed,dimension,window_size,epochs,workers,folds,cv_seed,walk_length,num_walks
0,0.300000,0,128,4,1,4,10,0,10,100


Completed pp=0 baseline with the original selector:
pp0_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,accuracy,0.809316,0.008050,0.792580,0.819409,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,auroc,0.892104,0.010099,0.871847,0.903323,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,aupr,0.898156,0.009165,0.879190,0.911426,10


Reused EM matrix: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_semantic_pp_mesh2024_roots/edge_transition_matrix.json
New walks: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_teleport_mesh2024_roots/walks.txt
New embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_teleport_mesh2024_roots/pp0.pkl
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_teleport_mesh2024_roots


### 8.1 Record configuration and baseline provenance

Check completed original-selector results, inputs, binary, EM matrix and folds. Use a separate run name.


In [67]:
run(TELEPORT_RUNNER, 'config', TELEPORT_RUN_NAME)

Running: run_teleport_mesh2024_control.py config | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/result

### 8.2 Prepare the binary, map and EM matrix

Reuse completed files without C++ compilation or EM retraining. Leave upstream source unchanged.


In [68]:
run(TELEPORT_RUNNER, 'prepare', TELEPORT_RUN_NAME)

Running: run_teleport_mesh2024_control.py prepare | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/resul

### 8.3 Generate first-hit-selector walks

Generate and save 2,995,900 length-10 walks. Worker records verify that macOS spawn workers actually call the diagnostic function.


In [69]:
run(TELEPORT_RUNNER, 'walks', TELEPORT_RUN_NAME)
print('Saved/reused: ', TELEPORT_DATA/'walks.txt')
worker_audit = read_json(TELEPORT_RESULTS/'worker_selector_audit.json')
show_rows([{key: worker_audit[key] for key in (
    'tasks', 'worker_pids', 'selector_calls', 'selector_source_sha256', 'rng_state_claim',
)}])


Running: run_teleport_mesh2024_control.py walks | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results

tasks,worker_pids,selector_calls,selector_source_sha256,rng_state_claim
100,"[84761, 84762, 84763, 84764]",325981,371fb63a4fff779563b7d379ea23b052e9207413d4aff17a0f916e8e07daffe9,No assertion about Python/NumPy RNG state inside workers


### 8.4 Original C++ pp=0: train and save immediately

Replace the corpus only. Retain trainer, Skip-gram settings and the pp=0 branch.


In [70]:
run(TELEPORT_RUNNER, 'embedding', TELEPORT_RUN_NAME)
print('Embedding saved/reused: ', TELEPORT_DATA/'pp0.pkl')

Running: run_teleport_mesh2024_control.py embedding | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/res

### 8.5 Same ten-fold XGBoost

Read saved embeddings, reuse negatives/order/subtraction features, and check fold IDs before fitting.


In [71]:
run(TELEPORT_RUNNER, 'evaluate', TELEPORT_RUN_NAME)

Running: run_teleport_mesh2024_control.py evaluate | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/resu

In [72]:
show_csv(TELEPORT_RESULTS/'pp0_summary.csv')

pp0_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,accuracy,0.831675,0.010039,0.814503,0.844726,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,auroc,0.911353,0.008327,0.896235,0.927285,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,semantic_same_walks_pp0,0,aupr,0.917174,0.009372,0.896007,0.933383,10


### 8.6 Compare selectors and the paper's SG reference

Inspect local **first-hit selector − original selector** metrics and paired fold deltas, both with pp=0.

The external SG reference is the arithmetic mean of ten rows in [Source Data](https://pmc.ncbi.nlm.nih.gov/articles/instance/10272215/bin/41467_2023_39301_MOESM4_ESM.xlsx), sheet `SupplementaryFigure5`. These are exact extracted values, not plot estimates, and do not replace the local baseline.


In [73]:
if not table_complete(
    TELEPORT_RESULTS/'teleport_selector_comparison.csv',
    TELEPORT_RESULTS/'comparison_complete.json',
    TELEPORT_RESULTS/'run_config.json',
):
    run(TELEPORT_RUNNER, 'compare', TELEPORT_RUN_NAME)
show_csv(TELEPORT_RESULTS/'teleport_selector_comparison.csv')
show_csv(TELEPORT_RESULTS/'paired_fold_deltas.csv')
PAPER_SKIPGRAM_REFERENCE = [
    {'model': 'Heterogeneous SG', 'pp': 1, 'n': 10,
     'accuracy': 0.8770982707718258, 'auroc': 0.9440604521100597,
     'aupr': 0.9467667350869341},
    {'model': 'SG', 'pp': 0, 'n': 10,
     'accuracy': 0.8688738928722053, 'auroc': 0.9385006510555634,
     'aupr': 0.9416743190886354},
]
show_rows(PAPER_SKIPGRAM_REFERENCE)

Running: run_teleport_mesh2024_control.py compare | msi_seed0_teleport_mesh2024_roots
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_semantic_pp_mesh2024_roots/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/resul

metric,original_selector_pp0,fixed_selector_pp0,fixed_minus_original
accuracy,0.809316,0.831675,0.022360
auroc,0.892104,0.911353,0.019249
aupr,0.898156,0.917174,0.019018


paired_fold_deltas.csv


fold,accuracy_fixed_minus_original,auroc_fixed_minus_original,aupr_fixed_minus_original
1,0.021922,0.024388,0.016817
2,0.016863,0.011867,0.015865
3,0.032911,0.028258,0.021956
4,0.027004,0.014115,0.009876
5,0.016034,0.022529,0.024025
6,0.025316,0.023233,0.018827
7,0.003376,0.003717,0.011663
8,0.027848,0.016655,0.024000
9,0.033755,0.029303,0.024909
10,0.018565,0.018424,0.022242


model,pp,n,accuracy,auroc,aupr
Heterogeneous SG,1,10,0.877098,0.944060,0.946767
SG,0,10,0.868874,0.938501,0.941674


## 9. Control 7: positive-output-type NS pool, τ=0

**Run Section 1, then this section; completed Sections 2–8 need not be repeated.**

**Question: on identical no-teleport walks, how does selecting negatives by positive-output type compare with input type?**

| Branch | Negative pool | Other stages |
|---|---|---|
| Completed baseline | Input node type | pp=1; Section 4 corpus and evaluation |
| Diagnostic | Positive output type | pp=1; identical corpus, map, pairs and folds |

This **DIAGNOSTIC IMPLEMENTATION CHANGE** follows the metapath2vec++ paper's output-type formulation. Both released metapath2vec++ and DREAMwalk C++ use input type. Do not label the variant as the authors' benchmark implementation.

Change only pp=1 pool selection and its matching type check: `last_word` → `word`. Keep frequency^0.75, negative=5, window 4, one epoch, 128D, four workers and original collision behavior. **Do not add positive-output rejection or change pp=0 here.**

Reuse the exact 2,995,900 length-10 walks. Save separate source, diff, binary, embedding and results without new EM/walks or baseline retraining.


In [11]:
NS_POOL_RUN_NAME = 'msi_seed0_ns_output_type_control'
NS_POOL_RUNNER = 'run_ns_output_type_control.py'
NS_POOL_RESULTS = RESULTS/NS_POOL_RUN_NAME
NS_POOL_DATA = ROOT/'data/embeddings'/NS_POOL_RUN_NAME
NS_POOL_REFERENCE_RESULTS = RESULTS/'msi_seed0_skipgram_pp_comparison'
NS_POOL_REFERENCE_DATA = ROOT/'data/embeddings/msi_seed0_skipgram_pp_comparison'

ns_pool_settings = runner_constants(ROOT/'src'/NS_POOL_RUNNER, 'RUN_SETTINGS')['RUN_SETTINGS']
check_parameters(ns_pool_settings, {
    'pp': 1, 'tp_factor': 0, 'seed': 0, 'num_walks': 100,
    'walk_length': 10, 'dimension': 128, 'window_size': 4,
    'epochs': 1, 'workers': 4, 'folds': 10, 'cv_seed': 0,
})
show_rows([ns_pool_settings])
print('Original input-type pp=1 baseline: ')
show_csv(NS_POOL_REFERENCE_RESULTS/'pp1_summary.csv')
print('Reused original corpus: ', NS_POOL_REFERENCE_DATA/'walks.txt')
print('Modified C++: ', NS_POOL_DATA/'HeterogeneousSG_output_type.cpp')
print('New embedding: ', NS_POOL_DATA/'pp1.pkl')
print('Results for this section: ', NS_POOL_RESULTS)

pp,tp_factor,seed,num_walks,walk_length,dimension,window_size,workers,epochs,negative,sample,folds,cv_seed,source_walks,source_nodes
1,0,0,100,10,128,4,4,1,5,0.001000,10,0,2995900,29959


Original input-type pp=1 baseline:
pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,accuracy,0.788137,0.011062,0.771308,0.808439,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,auroc,0.866120,0.007938,0.855043,0.880914,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,same_walks_pp1,0,aupr,0.874306,0.010921,0.860393,0.892279,10


Reused original corpus: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_skipgram_pp_comparison/walks.txt
Modified C++: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HeterogeneousSG_output_type.cpp
New embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/pp1.pkl
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_ns_output_type_control


### 9.1 Record configuration and original pp=1 provenance

Record hashes for corpus, map, source, baseline, pairs and folds. τ=0 bypasses the teleport selector.


In [12]:
run(NS_POOL_RUNNER, 'config', NS_POOL_RUN_NAME)

Running: run_ns_output_type_control.py config | msi_seed0_ns_output_type_control
{
  "settings": {
    "pp": 1,
    "tp_factor": 0,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "dimension": 128,
    "window_size": 4,
    "workers": 4,
    "epochs": 1,
    "negative": 5,
    "sample": 0.001,
    "folds": 10,
    "cv_seed": 0,
    "source_walks": 2995900,
    "source_nodes": 29959
  },
  "reference": {
    "fingerprint": "26b8c1b2a993ffe7ce83cafd3461536072e48fe77ef2e109bab79e8496b017ea",
    "inputs": {
      "graph": "8285845165826358275bc3dc379608a4e2de656b5398b5701123810eb51c61da",
      "similarity": "e46829cfb4013577e7488ffbc107ac0fbae1ceb352f9df01f972a48fe4426f7c",
      "nodetypes": "4e6898e31e3a06eb4d374b817c2a39cb42250534b5e6d46c0290fd0d1064e072",
      "positives": "5e8a402186092d94a5c576b754b12bb93ad90e4e51f851c20bba6e928bb7a5a4",
      "negatives": "f17ba4218f7bbc58b8bf4f9341eb53b55980dbf37faa9de17459bb64a3ee1a47"
    },
    "artifacts": {
      "corpus": "

### 9.2 Generate a local patch and separate binary

Check that the source diff changes only pp=1 pool type. Preserve original source, pp=0 and collision handling. Reuse corpus/map.


In [13]:
run(NS_POOL_RUNNER, 'prepare', NS_POOL_RUN_NAME)
print((NS_POOL_DATA/'patch.diff').read_text())

Running: run_ns_output_type_control.py prepare | msi_seed0_ns_output_type_control
Compiling independent output-type NS source; vendor remains unchanged
HeterogeneousSG_output_type.cpp:328:16: warning: variable 'i' set but not used [-Wunused-but-set-variable]
  328 |   long long a, i = 0;
      |                ^
1 warning generated.
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
--- vendor/DREAMwalk_upstream/DREAMwalk/HeterogeneousSG.cpp
+++ HeterogeneousSG_output_type.cpp
@@ -453,17 +453,17 @@
             next_random = next_random * (unsigned long long)25214903917 + 11;  
 
             if (pp) {
-              if        (vocab[last_word].word[0] == 'd') {
+              if       

### 9.3 Train output-type pp=1 and save immediately

Read exactly the baseline corpus. Train this single new embedding.


In [14]:
run(NS_POOL_RUNNER, 'embedding', NS_POOL_RUN_NAME)
print('Embedding saved/reused: ', NS_POOL_DATA/'pp1.pkl')

Running: run_ns_output_type_control.py embedding | msi_seed0_ns_output_type_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HetSG
Training: ./HetSG -train walks.txt -output pp1 -pp 1 -min-count 0 -size 128 -iter 1 -window 4 -threads 4
EMBEDDING SAVED: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_

### 9.4 Same ten-fold XGBoost

Read saved embeddings. Reuse evaluator/negatives and check pair order/fold IDs before fitting.


In [15]:
run(NS_POOL_RUNNER, 'evaluate', NS_POOL_RUN_NAME)

Running: run_ns_output_type_control.py evaluate | msi_seed0_ns_output_type_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/pp1.pkl
no_teleport negative seed 0 fold 1 {'accuracy': 0.779089, 'auroc': 0.859758, 'aupr': 0.865148}
no_teleport negative seed 0 fold 2 {'accuracy': 0.812816, 'auroc': 0.891636, 'aupr': 0.896111}
no_teleport negative seed 0 fold 3 {'accuracy': 0.794937, 'auroc': 0.887272, 'aupr': 0.900983}
no_teleport negative seed 0 fold 4 {'accuracy': 0.819409, 'auroc': 0.890593, 'aupr': 0.899431}
no_

In [16]:
show_csv(NS_POOL_RESULTS/'pp1_summary.csv')

pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,accuracy,0.800372,0.011957,0.779089,0.819409,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,auroc,0.881369,0.010521,0.859758,0.892365,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,aupr,0.890439,0.012071,0.865148,0.901075,10


### 9.5 Compare output-type and input-type NS

Inspect **output type − input type**. Both use pp=1, the same corpus and folds. The intended algorithm change is in the binary.

Section 4's pp=0 is an external local reference, not another name for this variant. Better performance supports the design on these inputs without proving author benchmark provenance.


In [17]:
if not table_complete(
    NS_POOL_RESULTS/'ns_pool_comparison.csv',
    NS_POOL_RESULTS/'comparison_complete.json',
    NS_POOL_RESULTS/'run_config.json',
):
    run(NS_POOL_RUNNER, 'compare', NS_POOL_RUN_NAME)
show_csv(NS_POOL_RESULTS/'ns_pool_comparison.csv')
show_csv(NS_POOL_RESULTS/'paired_fold_deltas.csv')

Running: run_ns_output_type_control.py compare | msi_seed0_ns_output_type_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/pp1.pkl
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_ns_output_type_control/pp1_summary.csv
NS POOL COMPARISON COMPLETE
[
  {
    "metric": "accuracy",
    "input_type_pp1": 0.7881374118584612,
    "output_type_pp1": 0.8003719910915675,
    "output_minus_input": 0.012234579233106335
  },
  {
    "

metric,input_type_pp1,output_type_pp1,output_minus_input
accuracy,0.788137,0.800372,0.012235
auroc,0.866120,0.881369,0.015249
aupr,0.874306,0.890439,0.016133


paired_fold_deltas.csv


fold,accuracy_output_minus_input,auroc_output_minus_input,aupr_output_minus_input
1,-0.004216,-0.005261,-0.009900
2,0.024452,0.018049,0.012821
3,0.018565,0.024150,0.026425
4,0.048101,0.035550,0.035764
5,0.004219,0.011841,0.010910
6,0.016034,0.011920,0.020302
7,-0.002532,0.011451,0.008796
8,0.003376,0.013049,0.017626
9,0.004219,0.015063,0.016621
10,0.010127,0.016681,0.021969


## 10. Control 8: reject the current positive output, output-type NS, τ=0

**Run Section 1, then this section. Earlier completed experiments need not be repeated.**

**Question: does skipping a negative update when it samples the current positive output improve performance?**

| Branch | Pool | Reject current positive output | Other settings |
|---|---|---|---|
| Section 9 baseline | Positive output type | No | pp=1; τ=0; same corpus and folds |
| Diagnostic | Positive output type | Yes | Same |

Section 9 retained collision behavior to isolate the pool endpoint. Add only:

```cpp
if (target == word) continue;
```

`word` is the current positive output. Apply this only in the negative branch; preserve the positive update. **Skip a colliding draw without resampling.** `negative=5` remains five attempts, possibly fewer than five updates, matching the original pp=0 equality-check style.

This does not consult all known positive pairs or exclude every context in the window. Keep output-type pools, frequency^0.75, subsampling=0.001 and all training/downstream settings fixed.

Reuse Section 9 corpus/map and completed embedding/evaluation as baseline. Save separate source, one-line patch, binary and outputs. No new EM/walks, baseline retraining or vendor edits. Label this **DIAGNOSTIC POSITIVE-OUTPUT FILTER**.


In [18]:
NS_FILTER_RUN_NAME = 'msi_seed0_ns_positive_filter_control'
NS_FILTER_RUNNER = 'run_ns_positive_filter_control.py'
NS_FILTER_RESULTS = RESULTS/NS_FILTER_RUN_NAME
NS_FILTER_DATA = ROOT/'data/embeddings'/NS_FILTER_RUN_NAME
NS_FILTER_REFERENCE_RESULTS = RESULTS/'msi_seed0_ns_output_type_control'
NS_FILTER_REFERENCE_DATA = ROOT/'data/embeddings/msi_seed0_ns_output_type_control'

ns_filter_settings = runner_constants(
    ROOT/'src'/NS_FILTER_RUNNER, 'RUN_SETTINGS')['RUN_SETTINGS']
check_parameters(ns_filter_settings, {
    'pp': 1, 'tp_factor': 0, 'seed': 0, 'num_walks': 100,
    'walk_length': 10, 'dimension': 128, 'window_size': 4,
    'epochs': 1, 'workers': 4, 'negative': 5, 'sample': 0.001,
    'folds': 10, 'cv_seed': 0, 'reject_positive_output': True,
    'rejection_policy': 'skip_without_resampling',
})
show_rows([ns_filter_settings])
print('Completed unfiltered output-type baseline: ')
show_csv(NS_FILTER_REFERENCE_RESULTS/'pp1_summary.csv')
print('Reused corpus: ', NS_FILTER_REFERENCE_DATA/'walks.txt')
print('New source: ', NS_FILTER_DATA/'HeterogeneousSG_output_type_reject_positive.cpp')
print('New embedding: ', NS_FILTER_DATA/'pp1.pkl')
print('Results for this section: ', NS_FILTER_RESULTS)

pp,tp_factor,seed,num_walks,walk_length,dimension,window_size,workers,epochs,negative,sample,folds,cv_seed,reject_positive_output,rejection_policy
1,0,0,100,10,128,4,4,1,5,0.001000,10,0,True,skip_without_resampling


Completed output-type baseline without collision filtering:
pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,accuracy,0.800372,0.011957,0.779089,0.819409,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,auroc,0.881369,0.010521,0.859758,0.892365,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_pp1,0,aupr,0.890439,0.012071,0.865148,0.901075,10


Reused corpus: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_output_type_control/walks.txt
New source: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HeterogeneousSG_output_type_reject_positive.cpp
New embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/pp1.pkl
Section results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_ns_positive_filter_control


### 10.1 Record configuration and baseline provenance

Check Section 9 completion records, source, binary, corpus, map, pairs and folds. Historical results are read-only.


In [19]:
run(NS_FILTER_RUNNER, 'config', NS_FILTER_RUN_NAME)

Running: run_ns_positive_filter_control.py config | msi_seed0_ns_positive_filter_control
{
  "settings": {
    "pp": 1,
    "tp_factor": 0,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "dimension": 128,
    "window_size": 4,
    "workers": 4,
    "epochs": 1,
    "negative": 5,
    "sample": 0.001,
    "folds": 10,
    "cv_seed": 0,
    "reject_positive_output": true,
    "rejection_policy": "skip_without_resampling"
  },
  "reference": {
    "fingerprint": "7336da6c8addef39b091efd5bddeb04b60738be31fc46e62ec3bb4f6d9a1bb3c",
    "inputs": {
      "graph": "8285845165826358275bc3dc379608a4e2de656b5398b5701123810eb51c61da",
      "similarity": "e46829cfb4013577e7488ffbc107ac0fbae1ceb352f9df01f972a48fe4426f7c",
      "nodetypes": "4e6898e31e3a06eb4d374b817c2a39cb42250534b5e6d46c0290fd0d1064e072",
      "positives": "5e8a402186092d94a5c576b754b12bb93ad90e4e51f851c20bba6e928bb7a5a4",
      "negatives": "f17ba4218f7bbc58b8bf4f9341eb53b55980dbf37faa9de17459bb64a3ee1a47"
    

### 10.2 Generate the one-line patch and separate binary

Add only the current-positive equality check to Section 9 source. The displayed diff must not change pools, pp=0 or parameters.


In [20]:
run(NS_FILTER_RUNNER, 'prepare', NS_FILTER_RUN_NAME)
print((NS_FILTER_DATA/'patch.diff').read_text())

Running: run_ns_positive_filter_control.py prepare | msi_seed0_ns_positive_filter_control
Compiling independent positive-output rejection control
HeterogeneousSG_output_type_reject_positive.cpp:328:16: warning: variable 'i' set but not used [-Wunused-but-set-variable]
  328 |   long long a, i = 0;
      |                ^
1 warning generated.
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/walks.txt
--- data/embeddings/msi_seed0_ns_output_type_control/HeterogeneousSG_output_type.cpp
+++ HeterogeneousSG_output_type_reject_positive.cpp
@@ -464,6 +464,7 @@
               }
               if (target == 0) target = next_random % (vocab_size - 1) + 1;
               if (vocab[word].word[0] !

### 10.3 Train filtered pp=1 and save immediately

Read the same corpus. Train one new embedding and save it before the next step.


In [21]:
run(NS_FILTER_RUNNER, 'embedding', NS_FILTER_RUN_NAME)
print('Embedding saved/reused: ', NS_FILTER_DATA/'pp1.pkl')

Running: run_ns_positive_filter_control.py embedding | msi_seed0_ns_positive_filter_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HetSG
Training: ./HetSG -train walks.txt -output pp1 -pp 1 -min-count 0 -size 128 -iter 1 -window 4 -threads 4
EMBEDDING SAVED: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Re

### 10.4 Same ten-fold XGBoost

Reuse negatives, pair order, subtraction features and classifier settings. Check fold IDs before fitting.


In [22]:
run(NS_FILTER_RUNNER, 'evaluate', NS_FILTER_RUN_NAME)

Running: run_ns_positive_filter_control.py evaluate | msi_seed0_ns_positive_filter_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/pp1.pkl
no_teleport negative seed 0 fold 1 {'accuracy': 0.780776, 'auroc': 0.867581, 'aupr': 0.872586}
no_teleport negative seed 0 fold 2 {'accuracy': 0.81113, 'auroc': 0.890499, 'aupr': 0.89592}
no_teleport negative seed 0 fold 3 {'accuracy': 0.806751, 'auroc': 0.891311, 'aupr': 0.905927}
no_teleport negative seed 0 fold 4 {'accuracy': 0.816034, 'auroc': 0.889419, 'au

In [23]:
show_csv(NS_FILTER_RESULTS/'pp1_summary.csv')

pp1_summary.csv


analysis,label,negative_seed,metric,mean,SD,min,max,n
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_filtered_pp1,0,accuracy,0.800119,0.010164,0.780776,0.816034,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_filtered_pp1,0,auroc,0.882343,0.010664,0.867581,0.898315,10
DIAGNOSTIC / SENSITIVITY ANALYSIS,no_teleport_output_type_filtered_pp1,0,aupr,0.890107,0.013022,0.872586,0.910121,10


### 10.5 Compare filtered and unfiltered output-type NS

Inspect **filtered − unfiltered** and fold deltas. Both branches use output-type pp=1, τ=0, identical corpus and folds.

Section 9 AUROC is 0.881369; Section 4 ordinary pp=0 is 0.881752. The latter is background, not the main comparator. The paper's without-teleport AUROC **0.904701** averages ten `Figure4b` Source Data rows. Report the external gap separately from the local filtering effect without score-driven tuning.


In [24]:
if not table_complete(
    NS_FILTER_RESULTS/'ns_positive_filter_comparison.csv',
    NS_FILTER_RESULTS/'comparison_complete.json',
    NS_FILTER_RESULTS/'run_config.json',
):
    run(NS_FILTER_RUNNER, 'compare', NS_FILTER_RUN_NAME)
show_csv(NS_FILTER_RESULTS/'ns_positive_filter_comparison.csv')
show_csv(NS_FILTER_RESULTS/'paired_fold_deltas.csv')

Running: run_ns_positive_filter_control.py compare | msi_seed0_ns_positive_filter_control
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_ns_positive_filter_control/pp1.pkl
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_ns_positive_filter_control/pp1_summary.csv
NS POSITIVE-OUTPUT FILTER COMPARISON COMPLETE
[
  {
    "metric": "accuracy",
    "output_type_unfiltered": 0.8003719910915675,
    "output_type_filtered": 0.8001188265346055,
    "f

metric,output_type_unfiltered,output_type_filtered,filtered_minus_unfiltered
accuracy,0.800372,0.800119,-0.000253
auroc,0.881369,0.882343,0.000974
aupr,0.890439,0.890107,-0.000332


paired_fold_deltas.csv


fold,accuracy_filtered_minus_unfiltered,auroc_filtered_minus_unfiltered,aupr_filtered_minus_unfiltered
1,0.001686,0.007823,0.007437
2,-0.001686,-0.001137,-0.000192
3,0.011814,0.004039,0.004944
4,-0.003376,-0.001174,-0.000908
5,-0.003376,-0.000863,-0.004764
6,-0.006751,-0.004062,-0.008620
7,-0.001688,0.005951,0.009046
8,-0.005063,0.001877,-0.004831
9,0.005907,-0.000632,0.000444
10,0.000000,-0.002085,-0.005877


## 11. MeSH2021: original DREAMwalk, τ=0.3, pp=1

**Run Section 1 → this section's parameter cell → 11.1–11.7. Sections 2–10 need not be run.**

**Question: what changes when only the MeSH Descriptor hierarchy year changes from 2024 to 2021 under original DREAMwalk?**

Supply `data/mesh/2021/d2021.bin`. Revalidate existing UMLS CUI-to-Descriptor candidates against 2021 records and Tree Numbers. This changes the **MeSH hierarchy year**, not the UMLS or ATC snapshot. Leave absent Descriptors unmapped; do not backfill 2024 ancestry or remove molecular nodes/evaluation pairs.

| Component | Setting |
|---|---|
| Molecular graph / ATC / labels / CV | Shared MSI inputs, saved seed-0 negatives and folds |
| MeSH hierarchy | 2021 Descriptor IDs and Tree Numbers; position-qualified nodes |
| Similarity | Original upstream IC/similarity, cutoff 0.4 |
| Walks | τ=0.3; 100 × 29,959; length 10; p=q=1; EM=5; four workers |
| Skip-gram | Original C++, pp=1; 128D; window 4; one epoch; four threads |
| Teleport / NS | Original logic; no diagnostic patches |
| Downstream | Same subtraction features and XGBoost |

Use separate hierarchy, similarity, corpus, embedding and result names. Save embeddings before evaluation. This historical experiment retains its original hierarchy construction; Section 12 contains the letter-root and combined variant.


In [ ]:
MESH2021_RUN_NAME = 'msi_seed0_mesh2021_original'
MESH2021_RUNNER = 'run_msi_mesh2021.py'
MESH2021_RESULTS = RESULTS/MESH2021_RUN_NAME
MESH2021_DATA = ROOT/'data/embeddings'/MESH2021_RUN_NAME
MESH2021_AUDITS = RESULTS/'msi_mesh2021_inputs'
MESH2021_SOURCE = ROOT/'data/mesh/2021/d2021.bin'

mesh2021_constants = runner_constants(
    ROOT/'src'/MESH2021_RUNNER, 'WALK_SETTINGS', 'SEMANTIC_SETTINGS', 'PP')
check_parameters(mesh2021_constants['WALK_SETTINGS'], {
    **EXPECTED_SHARED_WALKS, 'tp_factor': 0.3, 'walk_length': 10,
})
check_parameters(mesh2021_constants['SEMANTIC_SETTINGS'], {'cutoff': 0.4})
check_parameters({'pp': mesh2021_constants['PP']}, {'pp': 1})
mesh2021_sg_args = runner_constants(ROOT/'src/run_skipgram_pp_comparison.py', 'SG_ARGS')['SG_ARGS']
check_parameters(dict(zip(mesh2021_sg_args[::2], mesh2021_sg_args[1::2])), {
    '-size': '128', '-window': '4', '-iter': '1', '-threads': '4',
})
if not MESH2021_SOURCE.is_file():
    raise FileNotFoundError(MESH2021_SOURCE)
show_rows([{
    'MeSH': 2021, **mesh2021_constants['WALK_SETTINGS'],
    **mesh2021_constants['SEMANTIC_SETTINGS'], 'pp': mesh2021_constants['PP'],
}])
print('MeSH source：', MESH2021_SOURCE)
print('Mapping / hierarchy audits：', MESH2021_AUDITS)
print('New embedding: ', MESH2021_DATA/'pp1.pkl')
print('New results: ', MESH2021_RESULTS)


### 11.1 Validate MeSH2021 mapping and build hierarchy

Read 2021 Descriptor records and revalidate direct UMLS candidates. Reuse ATC edges. Retain `CUI → real Descriptor ID → DescriptorID@TreeNumber` paths. Save coverage, exclusions and position-node audits. No EM/walks/embeddings run here.


In [ ]:
run(MESH2021_RUNNER, 'hierarchy', MESH2021_RUN_NAME)
show_csv(MESH2021_AUDITS/'hierarchy_summary.csv')


### 11.2 Original semantic similarity, cutoff=0.4

Call upstream `save_sim_graph` on the new hierarchy and save a separate MeSH2021/ATC similarity file. Reuse completed computations; retain MeSH2024 inputs.


In [ ]:
run(MESH2021_RUNNER, 'similarity', MESH2021_RUN_NAME)
show_csv(MESH2021_AUDITS/'similarity_summary.csv')


### 11.3 Record configuration; prepare original binary and token map

Record raw source, hierarchy, similarity, shared inputs, source code and environment. Reuse Section 4's completed binary/map without rerunning Section 4 or editing upstream.


In [ ]:
run(MESH2021_RUNNER, 'config', MESH2021_RUN_NAME)


In [ ]:
run(MESH2021_RUNNER, 'prepare', MESH2021_RUN_NAME)


### 11.4 Original EM and MeSH2021 semantic walks

Generate 2,995,900 length-10 walks at τ=0.3. Retain original selector and RNG startup without an extra `PYTHONHASHSEED`. Save corpus/checkpoint for restart.


In [ ]:
run(MESH2021_RUNNER, 'walks', MESH2021_RUN_NAME)
print('Walks saved/reused: ', MESH2021_DATA/'walks.txt')


### 11.5 Original pp=1 embeddings, saved immediately

Read this MeSH2021 corpus with original heterogeneous Skip-gram. Save `pp1.pkl` before XGBoost.


In [ ]:
run(MESH2021_RUNNER, 'embedding', MESH2021_RUN_NAME)
print('Embedding saved/reused: ', MESH2021_DATA/'pp1.pkl')


### 11.6 Same negatives and ten-fold evaluation

Do not resample link-level negatives. Verify pair order and saved fold IDs before fitting. Each result is one seed-0 ten-fold repeat.


In [ ]:
run(MESH2021_RUNNER, 'evaluate', MESH2021_RUN_NAME)
show_csv(MESH2021_RESULTS/'pp1_summary.csv')


### 11.7 MeSH2021 − historical MeSH2024

Compare with original τ=0.3, pp=1 MeSH2024 results before letter roots and include Figure 2a references. Inspect `mesh2021_minus_mesh2024`. Years use separately sampled corpora but the same pair order, negatives, CV and model settings. This is one seed-0 contrast.


In [ ]:
run(MESH2021_RUNNER, 'compare', MESH2021_RUN_NAME)
show_csv(MESH2021_RESULTS/'mesh_year_comparison.csv')
print('Paper Figure 2a reference (mean of ten author repeats): ')
show_csv(RESULTS/'paper_fig2a_reference.csv')


## 12. Combined variant: seed 0, MeSH2024, τ=0.3, pp=1

**Run Section 1 → this section's parameter cell → 12.1–12.7. Earlier completed sections need not be run.**

| Stage | Implementation |
|---|---|
| Hierarchy | MeSH2024; separate letter-category nodes such as `C_dis`, `F_dis`, `G_dis` between top-level Tree Numbers and disease root, matching the demo structure |
| Teleport | Return at the first cumulative-weight hit; install/audit in every worker |
| pp=1 NS | Positive-output-type pool; skip a colliding current-positive draw without resampling |
| Shared inputs | Original MSI molecular graph, ATC, UMLS candidates, node types, link negatives and folds |
| Other settings | τ=0.3; cutoff 0.4; 100 walks/node; length 10; p=q=1; EM=5; 128D; window 4; one epoch; four workers/threads; same XGBoost |

Keep `DescriptorID@TreeNumber` position names. Add `_dis` to letter nodes to avoid collisions with ATC categories. Unmapped diseases remain in molecular graph/evaluation.

This is a **combined implementation variant**, not an isolated causal contrast. Earlier individual controls remain available. Its modified pp=1 differs from released pp=1.

Run name: `msi_seed0_mesh2024_final_fixed`. Use separate hierarchy, similarity, patched source/diff/binary, walks, embeddings and results. Save embeddings before evaluation. Use `data/mesh/2024/d2024.bin`; do not reuse MeSH2021 hierarchy/corpus/embedding.


In [42]:
FINAL_FIXED_RUN_NAME = 'msi_seed0_mesh2024_final_fixed'
FINAL_FIXED_RUNNER = 'run_msi_seed0_mesh2024_final_fixed.py'
FINAL_FIXED_RESULTS = RESULTS/FINAL_FIXED_RUN_NAME
FINAL_FIXED_DATA = ROOT/'data/embeddings'/FINAL_FIXED_RUN_NAME
FINAL_FIXED_AUDITS = RESULTS/'msi_mesh2024_demo_roots_inputs'

final_fixed_constants = runner_constants(
    ROOT/'src'/FINAL_FIXED_RUNNER, 'RUN_SETTINGS', 'WALK_SETTINGS', 'SEMANTIC_SETTINGS')
check_parameters(final_fixed_constants['RUN_SETTINGS'], {
    'seed': 0, 'tp_factor': 0.3, 'pp': 1, 'num_walks': 100,
    'walk_length': 10, 'dimension': 128, 'window_size': 4,
    'epochs': 1, 'workers': 4, 'folds': 10, 'cv_seed': 0,
    'ns_pool': 'positive_output_type', 'reject_positive_output': True,
    'rejection_policy': 'skip_without_resampling', 'teleport_selector': 'first_hit',
})
check_parameters(final_fixed_constants['WALK_SETTINGS'], {
    **EXPECTED_SHARED_WALKS, 'tp_factor': 0.3, 'walk_length': 10,
})
check_parameters(final_fixed_constants['SEMANTIC_SETTINGS'], {'mesh_year': 2024, 'cutoff': 0.4, 'letter_roots': True})
final_fixed_sg = runner_constants(ROOT/'src/run_skipgram_pp_comparison.py', 'SG_ARGS')['SG_ARGS']
check_parameters(dict(zip(final_fixed_sg[::2], final_fixed_sg[1::2])), {
    '-size': '128', '-window': '4', '-iter': '1', '-threads': '4',
})
show_rows([final_fixed_constants['RUN_SETTINGS']])
show_rows([final_fixed_constants['SEMANTIC_SETTINGS']])
print('MeSH source：', ROOT/'data/mesh/2024/d2024.bin')
print('New hierarchy/similarity audits: ', FINAL_FIXED_AUDITS)
print('Modified source/binary/walks/embedding: ', FINAL_FIXED_DATA)
print('Results: ', FINAL_FIXED_RESULTS)


pp,tp_factor,seed,num_walks,walk_length,dimension,window_size,workers,epochs,negative,sample,folds,cv_seed,ns_pool,reject_positive_output,rejection_policy,teleport_selector
1,0.300000,0,100,10,128,4,4,1,5,0.001000,10,0,positive_output_type,True,skip_without_resampling,first_hit


mesh_year,cutoff,letter_roots
2024,0.400000,True


MeSH source： /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/mesh/2024/d2024.bin
New hierarchy / similarity audits: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs
Corrected source / binary / walks / embedding: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed
Results: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_seed0_mesh2024_final_fixed


### 12.1 MeSH2024 hierarchy: add letter-category roots

Revalidate the same UMLS candidates against 2024 Tree Numbers. For example, `Descriptor@C13 → C_dis → disease`. Add each observed letter category. Save coverage/category/path audits without overwriting historical hierarchies.


In [43]:
run(FINAL_FIXED_RUNNER, 'hierarchy', FINAL_FIXED_RUN_NAME)
show_csv(FINAL_FIXED_AUDITS/'hierarchy_summary.csv')


Running: run_msi_seed0_mesh2024_final_fixed.py hierarchy | msi_seed0_mesh2024_final_fixed
{
  "stage": "hierarchy",
  "mesh_year": 2024,
  "mesh_descriptor_records": 30764,
  "mesh_descriptors_with_tree": 30762,
  "total_graph_diseases": 840,
  "hierarchy_covered_diseases": 748,
  "uncovered_diseases": 92,
  "total_graph_drugs": 1661,
  "hierarchy_covered_drugs": 1328,
  "uncovered_drugs": 333,
  "drug_hierarchy_edges": 7095,
  "disease_hierarchy_edges": 5464,
  "total_hierarchy_edges": 12559,
  "mesh_position_nodes": 2825,
  "disease_hierarchy_nodes": 4275,
  "unique_mesh_descriptor_uis": 1242,
  "max_disease_path_nodes": 13,
  "mesh_category_nodes": 4,
  "mesh_category_ids": [
    "A_dis",
    "C_dis",
    "F_dis",
    "G_dis"
  ],
  "disease_hierarchy_paths": 2048,
  "drug_hierarchy_paths": 2448,
  "mapping_status_counts": {
    "DIRECT_UNIQUE_USABLE": 748,
    "NO_MSH_MAPPING": 74,
    "SUPPLEMENTARY_UNRESOLVED": 18
  },
  "shared_drug_disease_hierarchy_node_ids": 0,
  "molecular_g

mesh_year,mesh_descriptor_records,mesh_descriptors_with_tree,total_graph_diseases,hierarchy_covered_diseases,uncovered_diseases,total_graph_drugs,hierarchy_covered_drugs,uncovered_drugs,drug_hierarchy_edges,disease_hierarchy_edges,total_hierarchy_edges,mesh_position_nodes,disease_hierarchy_nodes,unique_mesh_descriptor_uis,max_disease_path_nodes,mesh_category_nodes,mesh_category_ids,disease_hierarchy_paths,drug_hierarchy_paths,mapping_status_counts,shared_drug_disease_hierarchy_node_ids,molecular_graph_rebuilt,atc_hierarchy_changed,umls_rescanned
2024,30764,30762,840,748,92,1661,1328,333,7095,5464,12559,2825,4275,1242,13,4,"[""A_dis"", ""C_dis"", ""F_dis"", ""G_dis""]",2048,2448,"{""DIRECT_UNIQUE_USABLE"": 748, ""NO_MSH_MAPPING"": 74, ""SUPPLEMENTARY_UNRESOLVED"": 18}",0,False,False,False


### 12.2 Upstream IC/semantic similarity, cutoff=0.4

Apply unchanged upstream similarity code to the hierarchy with added categories and save a separate semantic graph. Keep ATC inputs, formula and cutoff fixed.


In [44]:
run(FINAL_FIXED_RUNNER, 'similarity', FINAL_FIXED_RUN_NAME)
show_csv(FINAL_FIXED_AUDITS/'similarity_summary.csv')


Running: run_msi_seed0_mesh2024_final_fixed.py similarity | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Similarity graph saved: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/processed/msi_mesh2024_demo_roots_similarity.tsv.lqxi4191
{
  "stage": "similarity",
  "cutoff": 0.4,
  "saved_edge_directionality": "one_row_per_undirected_pair",
  "semantic_pairs_by_type": {
    "disease": 25765,
    "drug": 27530
  },
  "nodes_with_semantic_edges_by_type": {
    "disease": 746,
    "drug": 1326
  },
  "total_semantic_pairs": 53295,
  "columns": [
    "node_1",
    "node_2",
    "semantic_type",
    "weight",
    "edge_index"
  ]
}
similarity_summary.csv


semantic_type,retained_undirected_pairs,nodes_with_semantic_edges,cutoff,saved_edge_directionality
drug,27530,1326,0.400000,one_row_per_undirected_pair
disease,25765,746,0.400000,one_row_per_undirected_pair


### 12.3 Configuration and patched C++ source/binary

Record input, source, patch and environment identities. Prepare output-type NS with positive-output rejection, the historical token map and same folds. Save source and diff separately from upstream.


In [45]:
run(FINAL_FIXED_RUNNER, 'config', FINAL_FIXED_RUN_NAME)


Running: run_msi_seed0_mesh2024_final_fixed.py config | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
{
  "settings": {
    "pp": 1,
    "tp_factor": 0.3,
    "seed": 0,
    "num_walks": 100,
    "walk_length": 10,
    "dimension": 128,
    "window_size": 4,
    "workers": 4,
    "epochs": 1,
    "negative": 5,
    "sample": 0.001,
    "folds": 10,
    "cv_seed": 0,
    "ns_pool": "positive_output_type",
    "reject_positive_output": true,
    "rejection_policy": "skip_without_resampling",
    "teleport_selector": "first_hit"
  },
  "walks": {
    "tp_factor": 0.3,
    "seed": 0,
    

In [46]:
run(FINAL_FIXED_RUNNER, 'prepare', FINAL_FIXED_RUN_NAME)


Running: run_msi_seed0_mesh2024_final_fixed.py prepare | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/HetSG
Patched pp=1 binary, exact token map and original saved fold IDs ready


### 12.4 Original EM → first-hit semantic walks

Use the new semantic graph and first-hit selector to generate 2,995,900 length-10 walks. Retain original seed=0 calls and multiprocessing start method, without an extra startup hash seed. Worker audits record selector installation and calls.


In [47]:
run(FINAL_FIXED_RUNNER, 'walks', FINAL_FIXED_RUN_NAME)
print('Walks saved/reused: ', FINAL_FIXED_DATA/'walks.txt')
final_fixed_worker_path = FINAL_FIXED_RESULTS/'worker_selector_audit.json'
if final_fixed_worker_path.is_file():
    final_fixed_worker_audit = read_json(final_fixed_worker_path)
    show_rows([{key: final_fixed_worker_audit.get(key) for key in (
        'tasks', 'worker_pids', 'selector_calls', 'selector_source_sha256', 'rng_state_claim',
    )}])


Running: run_msi_seed0_mesh2024_final_fixed.py walks | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/HetSG
random seed with 0
Starting unchanged edge-type EM: 5 iterations, length=10
Starting fixed MeSH2024 walks: tau=.3, 100 x 29,959, length=10
FINAL MESH2024 COMBINED CORPUS SAVED: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fix

tasks,worker_pids,selector_calls,selector_source_sha256,rng_state_claim
100,"[77912, 77913, 77914, 77915]",324356,371fb63a4fff779563b7d379ea23b052e9207413d4aff17a0f916e8e07daffe9,No assertion about Python/NumPy RNG state inside workers


### 12.5 Modified pp=1 embeddings, saved immediately

The binary uses positive-output-type pools and skips draws matching the current positive output. All other C++ parameters remain fixed. Atomically save `pp1.pkl` before this stage completes.


In [48]:
run(FINAL_FIXED_RUNNER, 'embedding', FINAL_FIXED_RUN_NAME)
print('Embedding saved/reused: ', FINAL_FIXED_DATA/'pp1.pkl')


Running: run_msi_seed0_mesh2024_final_fixed.py embedding | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddin

### 12.6 Same seed-0 pairs and ten-fold evaluation

Use the same 5,926 positives, 5,926 saved link negatives and folds. Verify pair order/fold IDs before fitting; retain subtraction features and XGBoost settings.


In [49]:
run(FINAL_FIXED_RUNNER, 'evaluate', FINAL_FIXED_RUN_NAME)
show_csv(FINAL_FIXED_RESULTS/'pp1_summary.csv')


Running: run_msi_seed0_mesh2024_final_fixed.py evaluate | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embedding

analysis,label,negative_seed,metric,mean,SD,min,max,n
"COMBINED DIAGNOSTIC: MeSH2024 letter roots, first-hit teleport, output-type NS with positive rejection",mesh2024_final_fixed_pp1,0,accuracy,0.835220,0.010732,0.816189,0.850633,10
"COMBINED DIAGNOSTIC: MeSH2024 letter roots, first-hit teleport, output-type NS with positive rejection",mesh2024_final_fixed_pp1,0,auroc,0.916308,0.009132,0.903304,0.928610,10
"COMBINED DIAGNOSTIC: MeSH2024 letter roots, first-hit teleport, output-type NS with positive rejection",mesh2024_final_fixed_pp1,0,aupr,0.922352,0.010408,0.905213,0.935841,10


### 12.7 Compare with historical original MeSH2024 and paper references

Inspect **combined MeSH2024 variant − historical original MeSH2024 τ=0.3, pp=1 before letter roots**, labeled `final_minus_original2024`. Also show Figure 2a. Section 13 keeps MeSH2021 results separate. Read optional combined MeSH2021 results only if their completed records exist.

Local rows are one seed-0 ten-fold repeat; paper references average ten repeats. This run combines hierarchy, teleport and NS changes and cannot separate their contributions. Its saved configuration/comparison retains the historical reference even after Sections 7–8 switch inputs.


In [50]:
run(FINAL_FIXED_RUNNER, 'compare', FINAL_FIXED_RUN_NAME)
show_csv(FINAL_FIXED_RESULTS/'final_fixed_comparison.csv')


Running: run_msi_seed0_mesh2024_final_fixed.py compare | msi_seed0_mesh2024_final_fixed
Reusing completed MeSH2024 category-layer hierarchy: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/hierarchy_manifest.json
Reusing completed MeSH2024 category-layer similarity: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/results/msi_mesh2024_demo_roots_inputs/similarity_manifest.json
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/HetSG
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings/msi_seed0_mesh2024_final_fixed/walks.txt
Reusing completed output: /Users/wangyuheng/projects/NUS/BMI5101/BMI5101_group/00.Reproduction/00.DREAMwalk_reproduction/data/embeddings

metric,original_mesh2024_pp1,final_mesh2024_fixed_pp1,final_minus_original2024,final_mesh2021_fixed_pp1,mesh2024_minus_mesh2021_final,paper_figure2a_mean,final_minus_paper
accuracy,0.807037,0.835220,0.028183,—,—,0.877098,-0.041879
auroc,0.887113,0.916308,0.029195,—,—,0.944060,-0.027753
aupr,0.895416,0.922352,0.026936,—,—,0.946767,-0.024415


## 13. Summary across experiments

Run Section 1 to read available results here. Separate original and diagnostic rows and use each experiment's actual result path/run name. Missing runs display blank values.

Section 11 is original MeSH2021 τ=0.3, pp=1; Section 12 is the combined MeSH2024 variant. Sections 9–10 both use output-type pp=1 and τ=0, differing only in current-positive rejection.

Paper references average ten repeats; local rows average ten folds for one seed. Sections 7–8 use revised MeSH2024 letter-root inputs and are shown separately from historical inputs. Rerun their parameter cells after updating a notebook with an existing kernel.


In [ ]:
# Run this summary after Section 1; it reads parameters and saved results only.
# Retain result paths from sections already configured with a different run name.
overview_semantic_settings = read_json(RESULTS/'msi_repeat0_run_config.json')['dreamwalk']
overview_semantic_rows = read_csv(RESULTS/'diagnostic_repeat0_comparison.csv')
overview_historical = summary_metrics(RESULTS/'diagnostic_no_teleport_summary.csv')
overview_length4 = runner_constants(ROOT/'src/run_no_teleport_length4.py', 'PARAMETERS')['PARAMETERS']
overview_no_teleport_walks = runner_constants(ROOT/'src/run_skipgram_pp_comparison.py', 'WALKS')['WALKS']
overview_edge_walks = runner_constants(ROOT/'src/run_edge2vec_walks_control.py', 'RUN_SETTINGS')['RUN_SETTINGS']
overview_gensim = read_json(ROOT.parent/'03.edge2vec_reproduction/results/msi_seed0/run_config.json')['config']
overview_semantic_walks = runner_constants(ROOT/'src/run_semantic_mesh2024_controls.py', 'WALK_SETTINGS')['WALK_SETTINGS']
overview_length4_results = globals().get('LENGTH4_RESULTS', RESULTS/'msi_seed0_no_teleport_length4')
overview_pp_results = globals().get('PP_RESULTS', RESULTS/'msi_seed0_skipgram_pp_comparison')
overview_edge_results = globals().get('EDGE_WALKS_RESULTS', RESULTS/'msi_seed0_edge2vec_walks_pp1')
overview_gensim_results = globals().get('GENSIM_RESULTS', RESULTS/'msi_seed0_edge2vec_gensim_xgb')
overview_semantic_results = globals().get('SEMANTIC_PP_RESULTS', RESULTS/'msi_seed0_semantic_pp_mesh2024_roots')
overview_historical_semantic_results = RESULTS/'msi_seed0_semantic_skipgram_pp_comparison'

overview_teleport_results = globals().get('TELEPORT_RESULTS', RESULTS/'msi_seed0_teleport_mesh2024_roots')
overview_historical_teleport_results = RESULTS/'msi_seed0_teleport_selector_control'
overview_ns_pool_results = globals().get('NS_POOL_RESULTS', RESULTS/'msi_seed0_ns_output_type_control')

overview_ns_filter_results = globals().get('NS_FILTER_RESULTS', RESULTS/'msi_seed0_ns_positive_filter_control')
overview_mesh2021_results = globals().get('MESH2021_RESULTS', RESULTS/'msi_seed0_mesh2021_original')
overview_final_fixed2021_results = RESULTS/'msi_seed0_mesh2021_final_fixed'
overview_final_fixed_results = globals().get('FINAL_FIXED_RESULTS', RESULTS/'msi_seed0_mesh2024_final_fixed')

semantic_metrics = {
    row['metric']: float(row['same_embedding_same_pairs_semantic_10fold'])
    for row in overview_semantic_rows
}
summary_cases = [
    ('Semantic seed0', overview_semantic_settings['tp_factor'], overview_semantic_settings['walk_length'], 1,
     semantic_metrics),
    ('No teleport — historical length10', 0, 10, 1, overview_historical),
    ('No teleport — length4', overview_length4['tp_factor'], overview_length4['walk_length'], 1,
     summary_metrics(overview_length4_results/'summary.csv')),
    ('Shared walks — pp1', overview_no_teleport_walks['tp_factor'], overview_no_teleport_walks['walk_length'], 1,
     summary_metrics(overview_pp_results/'pp1_summary.csv')),
    ('Shared walks — pp0', overview_no_teleport_walks['tp_factor'], overview_no_teleport_walks['walk_length'], 0,
     summary_metrics(overview_pp_results/'pp0_summary.csv')),
    ('Edge2vec walks → DREAMwalk pp1 + XGBoost', 0,
     overview_edge_walks['walk_length'], overview_edge_walks['pp'],
     summary_metrics(overview_edge_results/'pp1_summary.csv')),
    ('Edge2vec walks → Gensim SG + XGBoost', 0,
     overview_gensim['walk_length'], None,
     summary_metrics(overview_gensim_results/'summary.csv')),
    ('MeSH2024 + letter roots — original pp1', overview_semantic_walks['tp_factor'],
     overview_semantic_walks['walk_length'], 1,
     summary_metrics(overview_semantic_results/'pp1_summary.csv')),
    ('MeSH2024 + letter roots — original pp0', overview_semantic_walks['tp_factor'],
     overview_semantic_walks['walk_length'], 0,
     summary_metrics(overview_semantic_results/'pp0_summary.csv')),
    ('MeSH2024 + letter roots — corrected teleport pp0', 0.3, 10, 0,
     summary_metrics(overview_teleport_results/'pp0_summary.csv')),
    ('Historical MeSH2024 without letter roots — original pp1', 0.3, 10, 1,
     summary_metrics(overview_historical_semantic_results/'pp1_summary.csv')),
    ('Historical MeSH2024 without letter roots — original pp0', 0.3, 10, 0,
     summary_metrics(overview_historical_semantic_results/'pp0_summary.csv')),
    ('Historical MeSH2024 without letter roots — corrected teleport pp0', 0.3, 10, 0,
     summary_metrics(overview_historical_teleport_results/'pp0_summary.csv')),
    ('No teleport pp1 — output-type NS (diagnostic)', 0, 10, 1,
     summary_metrics(overview_ns_pool_results/'pp1_summary.csv')),
    ('No teleport pp1 — output-type NS + positive-output filter (diagnostic)', 0, 10, 1,
     summary_metrics(overview_ns_filter_results/'pp1_summary.csv')),
    ('MeSH2021 — original semantic pp1', 0.3, 10, 1,
     summary_metrics(overview_mesh2021_results/'pp1_summary.csv')),
    ('MeSH2021 — combined fixed tau0.3 pp1 (historical, if available)', 0.3, 10, 1,
     summary_metrics(overview_final_fixed2021_results/'pp1_summary.csv')),
    ('MeSH2024 — combined fixed tau0.3 pp1', 0.3, 10, 1,
     summary_metrics(overview_final_fixed_results/'pp1_summary.csv')),
    ('Paper — SG, SupplementaryFigure5', None, None, 0,
     {'accuracy': 0.8688738928722053, 'auroc': 0.9385006510555634,
      'aupr': 0.9416743190886354}),
    ('Paper — semantic, 10-repeat mean', None, None, 1,
     summary_metrics(RESULTS/'paper_fig2a_reference.csv')),
    ('Paper — no teleport, Figure4b mean', 0, None, 1,
     {'accuracy': 0.8318009278785319, 'auroc': 0.9047007990550796,
      'aupr': 0.9118934411514541}),
]
summary = [{
    'Experiment': name, 'tau': tau, 'walk_length': length, 'pp': pp,
    'Accuracy': metrics.get('accuracy'), 'AUROC': metrics.get('auroc'),
    'AUPR': metrics.get('aupr'),
} for name, tau, length, pp, metrics in summary_cases]
show_rows(summary)

## Extending these experiments

Add new controls for this model as sections here. Keep run names, parameters, checkpoints and results separate.

Combined variant route: **Section 1 → Section 12 → Section 13**. Semantic pp/selector route: **Section 1 → Section 7, including pp=0 evaluation → Section 8**. No-teleport controls, historical 2021 inputs and other databases do not switch hierarchy automatically.

Revisit accepted parameters only when a specific paper/code contradiction is identified. Historical records and duplicate notebook archives are omitted from this release; the repository README summarizes completed attempts. Colab retains its separate [notebook](../colab/DREAMwalk_MSI_seed0_Colab.ipynb).
